# QRG-ALNS-ILS S4–S5 v5 — Upload-Only, Two-Stage, Batched

This version uses only Colab uploads and downloads. It does not mount Google Drive. S4 is exported as a ZIP, every S5 batch is exported as a ZIP, and the merge stage reconstructs the final paper output from those downloaded ZIP files.


In [ ]:
# ============================================================
# 1. USER CONFIGURATION — V5 UPLOAD-ONLY PAPER PIPELINE
# ============================================================
# Run in this order:
#   1) STAGE = "build_s4"  -> once, for all 63 dates.
#   2) STAGE = "s5_batch"  -> repeat BATCH_ID = 1 ... 7.
#   3) STAGE = "merge"     -> once, after all batches finish.
#
# This version does NOT use Google Drive.
# State is preserved manually through downloaded ZIP files.
STAGE = "build_s4"  # "build_s4", "s5_batch", or "merge"

# Batch settings for STAGE="s5_batch".
BATCH_ID = 1         # 1..7
BATCH_SIZE = 9       # 7 batches cover 63 dates

OVERSIZE_POLICY = "split"
CAPACITY_WEIGHT_KG = 10_000.0
CAPACITY_VOLUME_CM3 = 3_992_800.0
DEPOT_START_MIN = 8 * 60
ROUTE_END_TARGET_MIN = 18 * 60
MAX_ROUTES_PER_DAY = 10
VEHICLE_OPENING_PENALTY = 1_000.0

# Calibrated stability penalty.
STABILITY_WEIGHT_FINAL = 120.0

# Stage 1: S4 is built once and locked.
S4_SEEDS = [11, 22, 33]
S4_ITERATIONS = 80
S4_LOCAL_SEARCH_TRIALS = 20
S4_EARLY_STOP = 20

# Stage 2: paper-grade S5 experiment.
S5_SEEDS = [101, 202, 303]
S5_TRIGGERS = (0.25, 0.50, 0.75)
S5_SHOCKS = ("p85", "p95", "delay20", "delay30")
S5_METHODS = ("NR", "FR", "RG", "SA")
S5_ITERATIONS = 40
S5_EARLY_STOP = 10

# Recompute behavior inside the current runtime.
FORCE_RECOMPUTE_S4 = False

# Leave None for interactive upload in Colab.
RAW_XLSX_PATH = None
MATRIX_ZIP_PATH = None

# Optional quick validation before the real run.
QUICK_TEST = False


In [ ]:
# ============================================================
# 2. ENVIRONMENT AND UPLOAD-ONLY INPUT HANDLING
# ============================================================
from pathlib import Path
import os, sys, json, shutil, zipfile, warnings, time, copy, hashlib
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", category=FutureWarning)

IN_COLAB = False
try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    files = None

BASE = Path("/content") if IN_COLAB else Path("/mnt/data")
WORK = BASE / "qrg_s4_s5_v5_upload_only_work"
LOCK_DIR = WORK / "locked_s4"
BATCH_DIR = WORK / "s5_batches"
CHECKPOINT_S4 = WORK / "checkpoints_s4"
CHECKPOINT_S5 = WORK / "checkpoints_s5"
FINAL_DIR = WORK / "paper_final_merged"
LOCAL_OUTPUT = BASE / "QRG_S4_S5_v5_upload_only_output"

for p in [WORK, LOCK_DIR, BATCH_DIR, CHECKPOINT_S4, CHECKPOINT_S5, FINAL_DIR, LOCAL_OUTPUT]:
    p.mkdir(parents=True, exist_ok=True)

if IN_COLAB:
    !pip -q install openpyxl xlsxwriter scipy

stage = STAGE.lower().strip()
if stage not in {"build_s4", "s5_batch", "merge"}:
    raise ValueError("STAGE must be 'build_s4', 's5_batch', or 'merge'.")

def upload_files_with_prompt(prompt_text):
    if not IN_COLAB:
        return {}
    print(prompt_text)
    uploaded = files.upload()
    return {name: BASE / name for name in uploaded}

def extract_zip(zip_path, target_dir):
    zip_path = Path(zip_path)
    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(target_dir)

def classify_uploads(uploaded_paths):
    paths = list(uploaded_paths.values())
    xlsx = [p for p in paths if p.suffix.lower() in {".xlsx", ".xls"}]
    zips = [p for p in paths if p.suffix.lower() == ".zip"]
    matrix = [p for p in zips if "matri" in p.name.lower() or "osrm" in p.name.lower()]
    locked = [p for p in zips if "locked" in p.name.lower()]
    batches = [p for p in zips if "batch" in p.name.lower()]
    return xlsx, zips, matrix, locked, batches

# Stage-specific uploads.
if stage == "build_s4":
    if RAW_XLSX_PATH is None or MATRIX_ZIP_PATH is None:
        if IN_COLAB:
            uploaded_paths = upload_files_with_prompt(
                "Upload BOTH files:\n"
                "1) baseline_dataset_prepared_coordinate_fixed.xlsx\n"
                "2) osrm_tomtom_saved_matrices (1).zip"
            )
            xlsx, zips, matrix, locked, batches = classify_uploads(uploaded_paths)
            if len(xlsx) != 1 or len(matrix) != 1:
                raise ValueError("Upload exactly one Excel dataset and one OSRM–TomTom matrix ZIP.")
            RAW_XLSX_PATH = str(xlsx[0])
            MATRIX_ZIP_PATH = str(matrix[0])
        else:
            default_xlsx = BASE / "baseline_dataset_prepared_coordinate_fixed.xlsx"
            default_zip = BASE / "osrm_tomtom_saved_matrices (1).zip"
            if not default_xlsx.exists() or not default_zip.exists():
                raise FileNotFoundError("Set RAW_XLSX_PATH and MATRIX_ZIP_PATH.")
            RAW_XLSX_PATH = str(default_xlsx)
            MATRIX_ZIP_PATH = str(default_zip)

elif stage == "s5_batch":
    if IN_COLAB:
        uploaded_paths = upload_files_with_prompt(
            "Upload THREE files:\n"
            "1) baseline_dataset_prepared_coordinate_fixed.xlsx\n"
            "2) osrm_tomtom_saved_matrices (1).zip\n"
            "3) QRG_v5_S4_Locked_Routes.zip"
        )
        xlsx, zips, matrix, locked, batches = classify_uploads(uploaded_paths)
        if len(xlsx) != 1:
            raise ValueError("Upload exactly one Excel dataset.")
        if len(matrix) != 1:
            raise ValueError("Could not identify the OSRM–TomTom matrix ZIP.")
        if len(locked) != 1:
            raise ValueError("Could not identify QRG_v5_S4_Locked_Routes.zip.")
        RAW_XLSX_PATH = str(xlsx[0])
        MATRIX_ZIP_PATH = str(matrix[0])
        extract_zip(locked[0], LOCK_DIR)
    else:
        raise RuntimeError("For local execution, place the input files manually.")

elif stage == "merge":
    if IN_COLAB:
        uploaded_paths = upload_files_with_prompt(
            "Upload these files together:\n"
            "- QRG_v5_S4_Locked_Routes.zip\n"
            "- QRG_v5_S5_Batch_01.zip through QRG_v5_S5_Batch_07.zip"
        )
        xlsx, zips, matrix, locked, batches = classify_uploads(uploaded_paths)
        if len(locked) != 1:
            raise ValueError("Upload exactly one S4 locked-route ZIP.")
        if len(batches) < 1:
            raise ValueError("Upload the completed S5 batch ZIP files.")
        extract_zip(locked[0], LOCK_DIR)
        for batch_zip in batches:
            extract_zip(batch_zip, BATCH_DIR)
    else:
        print("Local merge uses ZIP files already extracted into LOCK_DIR and BATCH_DIR.")

print("Stage:", stage)
print("Upload-only work folder:", WORK)


## 3. Core implementation


In [ ]:
from __future__ import annotations

import copy
import json
import math
import os
import random
import shutil
import time
import zipfile
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

DEPOT_ID = "DEPOT_1"
EPS = 1e-9


@dataclass
class Instance:
    delivery_date: str
    node_ids: List[str]
    weights: np.ndarray
    volumes: np.ndarray
    tw_start: np.ndarray
    tw_end: np.ndarray
    service: np.ndarray
    dist: np.ndarray
    tt50: np.ndarray
    tt85: np.ndarray
    tt95: np.ndarray

    @property
    def n_customers(self) -> int:
        return len(self.node_ids) - 1


@dataclass
class S4Config:
    capacity_weight_kg: float = 10000.0
    capacity_volume_cm3: float = 3992800.0
    depot_start_min: float = 480.0
    route_end_target_min: float = 1080.0
    max_routes: int = 20
    theta1_lateness_min: float = 0.0
    theta2_lateness_min: float = 15.0
    w_distance: float = 1.0
    w_travel: float = 0.15
    w_lateness: float = 7.0
    w_risk: float = 0.60
    w_vehicle: float = 250.0
    w_capacity_imbalance: float = 25.0
    w_overtime: float = 8.0
    iterations: int = 80
    destroy_min_frac: float = 0.08
    destroy_max_frac: float = 0.22
    local_search_trials: int = 20
    segment_length: int = 20
    reaction_factor: float = 0.25
    initial_temperature_ratio: float = 0.05
    cooling_rate: float = 0.995
    early_stop_patience: int = 25
    early_stop_min_iterations: int = 20


@dataclass
class S5Config:
    depot_start_min: float = 480.0
    route_end_target_min: float = 1080.0
    trigger_fractions: Tuple[float, ...] = (0.25, 0.50, 0.75)
    shocks: Tuple[str, ...] = ("p85", "p95", "delay10", "delay20", "delay30")
    methods: Tuple[str, ...] = ("NR", "DP", "FR", "RG", "SA")
    iterations_fr: int = 100
    iterations_rg: int = 100
    iterations_sa: int = 100
    w_distance: float = 1.0
    w_travel: float = 0.15
    w_lateness: float = 8.0
    w_overtime: float = 8.0
    w_risk: float = 0.75
    stability_weight: float = 20.0
    initial_temperature_ratio: float = 0.04
    cooling_rate: float = 0.992
    early_stop_patience: int = 25
    early_stop_min_iterations: int = 20


class DataRepository:
    REQUIRED = {
        "stops": "stops.csv",
        "distance": "combined_osrm_distance_matrix_km_long.csv",
        "p50": "combined_travel_time_matrix_p50_long.csv",
        "p85": "combined_travel_time_matrix_p85_long.csv",
        "p95": "combined_travel_time_matrix_p95_long.csv",
    }

    def __init__(self, input_path: str | Path, workdir: str | Path,
                 capacity_weight_kg: float = 10000.0,
                 capacity_volume_cm3: float = 3992800.0,
                 oversize_policy: str = "split"):
        self.input_path = Path(input_path)
        self.capacity_weight_kg = float(capacity_weight_kg)
        self.capacity_volume_cm3 = float(capacity_volume_cm3)
        self.oversize_policy = str(oversize_policy).lower()
        if self.oversize_policy not in {"split", "error"}:
            raise ValueError("oversize_policy must be 'split' or 'error'")
        self.workdir = Path(workdir)
        self.workdir.mkdir(parents=True, exist_ok=True)
        self.root = self._prepare_input()
        self.stops = self._read_stops(self.root / self.REQUIRED["stops"])
        self.matrix_frames = {
            "distance": self._read_matrix(self.root / self.REQUIRED["distance"], "distance_km"),
            "p50": self._read_matrix(self.root / self.REQUIRED["p50"], "travel_time_min"),
            "p85": self._read_matrix(self.root / self.REQUIRED["p85"], "travel_time_min"),
            "p95": self._read_matrix(self.root / self.REQUIRED["p95"], "travel_time_min"),
        }
        self._groups = {
            name: frame.groupby("delivery_date", sort=False).groups
            for name, frame in self.matrix_frames.items()
        }
        self.dates = sorted(set(self.stops["delivery_date"]))
        self._validate_date_coverage()

    def _prepare_input(self) -> Path:
        if self.input_path.is_dir():
            root = self.input_path
        elif self.input_path.suffix.lower() == ".zip":
            root = self.workdir / "input_extracted"
            if root.exists():
                shutil.rmtree(root)
            root.mkdir(parents=True)
            with zipfile.ZipFile(self.input_path) as zf:
                zf.extractall(root)
        else:
            raise ValueError(f"Input must be a folder or ZIP: {self.input_path}")

        # Locate required files even when nested.
        resolved = {}
        for key, fn in self.REQUIRED.items():
            hits = list(root.rglob(fn))
            if not hits:
                raise FileNotFoundError(f"Required input not found: {fn}")
            resolved[key] = hits[0]
        common = self.workdir / "normalized_input"
        common.mkdir(exist_ok=True)
        for key, src in resolved.items():
            dst = common / self.REQUIRED[key]
            if src.resolve() != dst.resolve():
                shutil.copy2(src, dst)
        return common

    @staticmethod
    def _normalize_date(series: pd.Series) -> pd.Series:
        return pd.to_datetime(series, errors="raise").dt.strftime("%Y-%m-%d")

    def _read_stops(self, path: Path) -> pd.DataFrame:
        df = pd.read_csv(path)
        required = {
            "delivery_date", "node_id", "weight_kg", "volume_cm3",
            "tw_start_min", "tw_end_min", "service_time_min"
        }
        missing = required - set(df.columns)
        if missing:
            raise ValueError(f"stops.csv missing columns: {sorted(missing)}")
        df = df[list(required)].copy()
        df["delivery_date"] = self._normalize_date(df["delivery_date"])
        df["node_id"] = df["node_id"].astype(str)
        for c in ["weight_kg", "volume_cm3", "tw_start_min", "tw_end_min", "service_time_min"]:
            df[c] = pd.to_numeric(df[c], errors="raise")
        if df.duplicated(["delivery_date", "node_id"]).any():
            dups = df[df.duplicated(["delivery_date", "node_id"], keep=False)].head()
            raise ValueError(f"Duplicate date-node rows in stops.csv:\n{dups}")
        if (df["tw_end_min"] < df["tw_start_min"]).any():
            raise ValueError("Some time windows end before they start.")
        return df.sort_values(["delivery_date", "node_id"]).reset_index(drop=True)

    def _read_matrix(self, path: Path, value_col: str) -> pd.DataFrame:
        df = pd.read_csv(path)
        required = {"delivery_date", "origin_id", "destination_id", value_col}
        missing = required - set(df.columns)
        if missing:
            raise ValueError(f"{path.name} missing columns: {sorted(missing)}")
        df = df[list(required)].copy()
        df["delivery_date"] = self._normalize_date(df["delivery_date"])
        df["origin_id"] = df["origin_id"].astype(str)
        df["destination_id"] = df["destination_id"].astype(str)
        df[value_col] = pd.to_numeric(df[value_col], errors="raise")
        if df.duplicated(["delivery_date", "origin_id", "destination_id"]).any():
            raise ValueError(f"Duplicate OD rows in {path.name}")
        return df

    def _validate_date_coverage(self):
        stop_dates = set(self.dates)
        errors = []
        for name, groups in self._groups.items():
            missing = stop_dates - set(groups)
            if missing:
                errors.append(f"{name}: missing {len(missing)} dates")
        if errors:
            raise ValueError("Matrix date coverage incomplete: " + "; ".join(errors))

    def audit(self) -> pd.DataFrame:
        rows = []
        for date, sdf in self.stops.groupby("delivery_date"):
            n = len(sdf) + 1
            split_parts = np.maximum(
                np.ceil(sdf["weight_kg"].to_numpy(float) / self.capacity_weight_kg),
                np.ceil(sdf["volume_cm3"].to_numpy(float) / self.capacity_volume_cm3),
            ).astype(int)
            row = {
                "delivery_date": date, "customer_count_raw": len(sdf),
                "oversize_stop_count": int(np.sum(split_parts > 1)),
                "customer_count_after_split": int(split_parts.sum()),
                "expected_od_raw": n*n, "oversize_policy": self.oversize_policy,
            }
            for name, frame in self.matrix_frames.items():
                idx = self._groups[name][date]
                row[f"{name}_od"] = len(idx)
                row[f"{name}_complete"] = len(idx) == n*n
            rows.append(row)
        return pd.DataFrame(rows)

    def _apply_oversize_policy(self, inst: Instance) -> Instance:
        parts = np.maximum(
            np.ceil(inst.weights[1:] / self.capacity_weight_kg),
            np.ceil(inst.volumes[1:] / self.capacity_volume_cm3),
        ).astype(int)
        if np.all(parts <= 1):
            return inst
        bad = [(inst.node_ids[i+1], int(parts[i]), float(inst.weights[i+1]), float(inst.volumes[i+1]))
               for i in range(len(parts)) if parts[i] > 1]
        if self.oversize_policy == "error":
            raise ValueError(
                f"Instance {inst.delivery_date} has stops exceeding vehicle capacity: {bad}. "
                "Use oversize_policy='split' only if split delivery is academically accepted."
            )
        source_indices = [0]
        node_ids = [DEPOT_ID]
        weights = [0.0]; volumes = [0.0]; tw_start = [0.0]; tw_end = [1e9]; service = [0.0]
        split_group = [None]
        for orig in range(1, len(inst.node_ids)):
            k = int(parts[orig-1])
            for part in range(1, k+1):
                source_indices.append(orig)
                node_ids.append(inst.node_ids[orig] if k == 1 else f"{inst.node_ids[orig]}__SPLIT{part}of{k}")
                weights.append(inst.weights[orig] / k)
                volumes.append(inst.volumes[orig] / k)
                tw_start.append(inst.tw_start[orig]); tw_end.append(inst.tw_end[orig])
                service.append(inst.service[orig] / k)
                split_group.append(inst.node_ids[orig])
        src = np.asarray(source_indices, dtype=int)
        def expand(mat):
            out = mat[np.ix_(src, src)].copy()
            # Virtual parts represent the same physical location.
            for a in range(1, len(src)):
                for b in range(1, len(src)):
                    if a != b and src[a] == src[b]:
                        out[a, b] = 0.0
            return out
        return Instance(
            delivery_date=inst.delivery_date, node_ids=node_ids,
            weights=np.asarray(weights), volumes=np.asarray(volumes),
            tw_start=np.asarray(tw_start), tw_end=np.asarray(tw_end), service=np.asarray(service),
            dist=expand(inst.dist), tt50=expand(inst.tt50), tt85=expand(inst.tt85), tt95=expand(inst.tt95),
        )

    def get_instance(self, delivery_date: str) -> Instance:
        date = pd.to_datetime(delivery_date).strftime("%Y-%m-%d")
        sdf = self.stops[self.stops["delivery_date"] == date].copy()
        if sdf.empty:
            raise KeyError(f"No stops for {date}")
        node_ids = [DEPOT_ID] + sdf["node_id"].tolist()
        n = len(node_ids)

        def dense(name: str, value_col: str) -> np.ndarray:
            frame = self.matrix_frames[name]
            idx = self._groups[name][date]
            sub = frame.loc[idx]
            mat = sub.pivot(index="origin_id", columns="destination_id", values=value_col)
            mat = mat.reindex(index=node_ids, columns=node_ids)
            if mat.isna().any().any():
                missing = int(mat.isna().sum().sum())
                raise ValueError(f"{name} matrix for {date} has {missing} missing OD values")
            return mat.to_numpy(dtype=float)

        zeros = np.zeros(1, dtype=float)
        inst = Instance(
            delivery_date=date,
            node_ids=node_ids,
            weights=np.concatenate([zeros, sdf["weight_kg"].to_numpy(float)]),
            volumes=np.concatenate([zeros, sdf["volume_cm3"].to_numpy(float)]),
            tw_start=np.concatenate([[0.0], sdf["tw_start_min"].to_numpy(float)]),
            tw_end=np.concatenate([[1e9], sdf["tw_end_min"].to_numpy(float)]),
            service=np.concatenate([zeros, sdf["service_time_min"].to_numpy(float)]),
            dist=dense("distance", "distance_km"),
            tt50=dense("p50", "travel_time_min"),
            tt85=dense("p85", "travel_time_min"),
            tt95=dense("p95", "travel_time_min"),
        )
        return self._apply_oversize_policy(inst)


def capacity_feasible(route: Sequence[int], inst: Instance, cfg: S4Config) -> bool:
    if not route:
        return True
    idx = np.asarray(route, dtype=int)
    return (
        inst.weights[idx].sum() <= cfg.capacity_weight_kg + EPS
        and inst.volumes[idx].sum() <= cfg.capacity_volume_cm3 + EPS
    )


def _adaptive_leg(current_time: float, i: int, j: int, inst: Instance, cfg: S4Config) -> Tuple[float, str, float]:
    arr50 = current_time + inst.tt50[i, j]
    predicted_late = max(0.0, arr50 - inst.tw_end[j])
    if predicted_late <= cfg.theta1_lateness_min:
        return inst.tt50[i, j], "p50", 0.0
    if predicted_late <= cfg.theta2_lateness_min:
        extra = inst.tt85[i, j] - inst.tt50[i, j]
        return inst.tt85[i, j], "p85", max(0.0, extra)
    extra = inst.tt95[i, j] - inst.tt50[i, j]
    return inst.tt95[i, j], "p95", max(0.0, extra)


def evaluate_route(
    route: Sequence[int], inst: Instance, cfg: S4Config,
    mode: str = "adaptive", start_time: Optional[float] = None,
    start_node: int = 0, include_return: bool = True,
) -> Dict:
    t = cfg.depot_start_min if start_time is None else float(start_time)
    prev = int(start_node)
    total_distance = 0.0
    total_travel = 0.0
    total_lateness = 0.0
    total_wait = 0.0
    risk_penalty = 0.0
    on_time = 0
    details = []
    q_counts = {"p50": 0, "p85": 0, "p95": 0}

    fixed_map = {"p50": inst.tt50, "p85": inst.tt85, "p95": inst.tt95}
    for node in route:
        node = int(node)
        if mode == "adaptive":
            tt, q, risk_extra = _adaptive_leg(t, prev, node, inst, cfg)
        else:
            tt = fixed_map[mode][prev, node]
            q = mode
            risk_extra = max(0.0, tt - inst.tt50[prev, node])
        q_counts[q] += 1
        total_distance += inst.dist[prev, node]
        total_travel += tt
        risk_penalty += risk_extra
        raw_arrival = t + tt
        waiting = max(0.0, inst.tw_start[node] - raw_arrival)
        arrival = raw_arrival + waiting
        late = max(0.0, arrival - inst.tw_end[node])
        total_wait += waiting
        total_lateness += late
        if late <= EPS:
            on_time += 1
        departure = arrival + inst.service[node]
        details.append({
            "node": node, "arrival_min": arrival, "departure_min": departure,
            "waiting_min": waiting, "lateness_min": late, "quantile": q,
        })
        t = departure
        prev = node

    if include_return:
        if mode == "adaptive":
            # Depot has no time window, so return is evaluated at p50.
            tt = inst.tt50[prev, 0]
        else:
            tt = fixed_map[mode][prev, 0]
        total_distance += inst.dist[prev, 0]
        total_travel += tt
        t += tt

    idx = np.asarray(route, dtype=int) if route else np.asarray([], dtype=int)
    weight = float(inst.weights[idx].sum()) if len(idx) else 0.0
    volume = float(inst.volumes[idx].sum()) if len(idx) else 0.0
    util_w = weight / cfg.capacity_weight_kg
    util_v = volume / cfg.capacity_volume_cm3
    overtime = max(0.0, t - cfg.route_end_target_min)
    return {
        "distance_km": total_distance,
        "travel_time_min": total_travel,
        "lateness_min": total_lateness,
        "waiting_min": total_wait,
        "on_time_count": on_time,
        "customer_count": len(route),
        "risk_penalty_min": risk_penalty,
        "route_end_min": t,
        "overtime_min": overtime,
        "weight_kg": weight,
        "volume_cm3": volume,
        "weight_util": util_w,
        "volume_util": util_v,
        "capacity_imbalance": abs(util_w - util_v),
        "quantile_counts": q_counts,
        "details": details,
    }


def route_objective(metrics: Dict, cfg: S4Config) -> float:
    return (
        cfg.w_distance * metrics["distance_km"]
        + cfg.w_travel * metrics["travel_time_min"]
        + cfg.w_lateness * metrics["lateness_min"]
        + cfg.w_risk * metrics["risk_penalty_min"]
        + cfg.w_capacity_imbalance * metrics["capacity_imbalance"]
        + cfg.w_overtime * metrics["overtime_min"]
    )


def evaluate_solution(routes: Sequence[Sequence[int]], inst: Instance, cfg: S4Config, mode: str = "adaptive") -> Dict:
    route_metrics = [evaluate_route(r, inst, cfg, mode=mode) for r in routes if r]
    totals = {
        "distance_km": sum(m["distance_km"] for m in route_metrics),
        "travel_time_min": sum(m["travel_time_min"] for m in route_metrics),
        "lateness_min": sum(m["lateness_min"] for m in route_metrics),
        "waiting_min": sum(m["waiting_min"] for m in route_metrics),
        "risk_penalty_min": sum(m["risk_penalty_min"] for m in route_metrics),
        "overtime_min": sum(m["overtime_min"] for m in route_metrics),
        "on_time_count": sum(m["on_time_count"] for m in route_metrics),
        "customer_count": sum(m["customer_count"] for m in route_metrics),
        "vehicle_count": len(route_metrics),
        "capacity_imbalance": sum(m["capacity_imbalance"] for m in route_metrics),
        "route_metrics": route_metrics,
    }
    totals["otd_pct"] = 100.0 * totals["on_time_count"] / max(1, totals["customer_count"])
    totals["objective"] = (
        sum(route_objective(m, cfg) for m in route_metrics)
        + cfg.w_vehicle * len(route_metrics)
    )
    return totals


def _route_delta(route: List[int], new_route: List[int], inst: Instance, cfg: S4Config) -> float:
    old = route_objective(evaluate_route(route, inst, cfg), cfg) if route else 0.0
    new = route_objective(evaluate_route(new_route, inst, cfg), cfg)
    return new - old


def best_insertion(customer: int, routes: List[List[int]], inst: Instance, cfg: S4Config) -> Optional[Tuple[float, int, int]]:
    options: List[Tuple[float, int, int]] = []
    for r_idx, route in enumerate(routes):
        if not capacity_feasible(route + [customer], inst, cfg):
            continue
        for pos in range(len(route) + 1):
            nr = route[:pos] + [customer] + route[pos:]
            delta = _route_delta(route, nr, inst, cfg)
            options.append((delta, r_idx, pos))
    if len(routes) < cfg.max_routes:
        if capacity_feasible([customer], inst, cfg):
            delta = route_objective(evaluate_route([customer], inst, cfg), cfg) + cfg.w_vehicle
            options.append((delta, len(routes), 0))
    if not options:
        return None
    return min(options, key=lambda x: x[0])


def construct_initial_solution(inst: Instance, cfg: S4Config) -> List[List[int]]:
    customers = list(range(1, inst.n_customers + 1))
    # Earliest due date first, then high volume ratio to avoid late capacity dead-ends.
    customers.sort(key=lambda i: (inst.tw_end[i], -inst.volumes[i] / cfg.capacity_volume_cm3, -inst.weights[i]))
    routes: List[List[int]] = []
    for c in customers:
        ins = best_insertion(c, routes, inst, cfg)
        if ins is None:
            raise RuntimeError(f"Cannot insert customer {inst.node_ids[c]} within capacity/max_routes")
        _, r_idx, pos = ins
        if r_idx == len(routes):
            routes.append([c])
        else:
            routes[r_idx].insert(pos, c)
    return [r for r in routes if r]


def _customer_lateness_map(routes: List[List[int]], inst: Instance, cfg: S4Config) -> Dict[int, float]:
    out = {}
    for route in routes:
        m = evaluate_route(route, inst, cfg)
        for d in m["details"]:
            out[d["node"]] = d["lateness_min"]
    return out


def destroy_solution(routes: List[List[int]], inst: Instance, cfg: S4Config, q: int, op: str, rng: random.Random) -> Tuple[List[List[int]], List[int]]:
    work = [r[:] for r in routes]
    all_customers = [c for r in work for c in r]
    q = min(q, len(all_customers))
    if q <= 0:
        return work, []

    if op == "random":
        removed = rng.sample(all_customers, q)
    elif op == "worst_lateness":
        late = _customer_lateness_map(work, inst, cfg)
        ranked = sorted(all_customers, key=lambda c: late.get(c, 0.0) + rng.random() * 0.01, reverse=True)
        removed = ranked[:q]
    elif op == "related":
        seed = rng.choice(all_customers)
        rel = []
        for c in all_customers:
            demand_gap = abs(inst.volumes[c] - inst.volumes[seed]) / max(cfg.capacity_volume_cm3, 1.0)
            tw_gap = abs(inst.tw_end[c] - inst.tw_end[seed]) / 240.0
            score = inst.dist[seed, c] / 20.0 + tw_gap + demand_gap + rng.random() * 0.05
            rel.append((score, c))
        removed = [c for _, c in sorted(rel)[:q]]
    elif op == "route":
        target = rng.choice([r for r in work if r])
        if len(target) >= q:
            removed = rng.sample(target, q)
        else:
            removed = target[:] + rng.sample([c for c in all_customers if c not in target], q - len(target))
    else:
        raise ValueError(op)

    remset = set(removed)
    work = [[c for c in r if c not in remset] for r in work]
    work = [r for r in work if r]
    return work, removed


def _all_insertions(customer: int, routes: List[List[int]], inst: Instance, cfg: S4Config) -> List[Tuple[float, int, int]]:
    opts = []
    for r_idx, route in enumerate(routes):
        if not capacity_feasible(route + [customer], inst, cfg):
            continue
        for pos in range(len(route) + 1):
            nr = route[:pos] + [customer] + route[pos:]
            opts.append((_route_delta(route, nr, inst, cfg), r_idx, pos))
    if len(routes) < cfg.max_routes and capacity_feasible([customer], inst, cfg):
        opts.append((route_objective(evaluate_route([customer], inst, cfg), cfg) + cfg.w_vehicle, len(routes), 0))
    return sorted(opts, key=lambda x: x[0])


def repair_solution(routes: List[List[int]], removed: List[int], inst: Instance, cfg: S4Config, op: str, rng: random.Random) -> List[List[int]]:
    work = [r[:] for r in routes]
    pool = removed[:]
    if op == "urgency":
        pool.sort(key=lambda c: (inst.tw_end[c], -inst.volumes[c]))
        mode = "greedy"
    else:
        rng.shuffle(pool)
        mode = op

    while pool:
        if mode == "regret2":
            candidates = []
            for c in pool:
                opts = _all_insertions(c, work, inst, cfg)
                if not opts:
                    continue
                regret = (opts[1][0] - opts[0][0]) if len(opts) > 1 else 1e6
                candidates.append((regret, -opts[0][0], c, opts[0]))
            if not candidates:
                raise RuntimeError("Repair failed: no feasible insertion")
            _, _, c, best = max(candidates)
        else:
            candidates = []
            for c in pool:
                opts = _all_insertions(c, work, inst, cfg)
                if opts:
                    candidates.append((opts[0][0], c, opts[0]))
            if not candidates:
                raise RuntimeError("Repair failed: no feasible insertion")
            _, c, best = min(candidates, key=lambda x: x[0])
        _, r_idx, pos = best
        if r_idx == len(work):
            work.append([c])
        else:
            work[r_idx].insert(pos, c)
        pool.remove(c)
    return [r for r in work if r]


def _random_local_move(routes: List[List[int]], inst: Instance, cfg: S4Config, rng: random.Random) -> Optional[List[List[int]]]:
    if not routes:
        return None
    move = rng.choice(["relocate", "swap", "2opt"])
    cand = [r[:] for r in routes]
    if move == "2opt":
        eligible = [i for i, r in enumerate(cand) if len(r) >= 4]
        if not eligible:
            return None
        ri = rng.choice(eligible)
        a, b = sorted(rng.sample(range(len(cand[ri])), 2))
        if b - a < 2:
            return None
        cand[ri][a:b+1] = reversed(cand[ri][a:b+1])
    elif move == "relocate":
        source_ids = [i for i, r in enumerate(cand) if r]
        if not source_ids:
            return None
        si = rng.choice(source_ids)
        pos = rng.randrange(len(cand[si]))
        c = cand[si].pop(pos)
        target_choices = list(range(len(cand))) + ([len(cand)] if len(cand) < cfg.max_routes else [])
        ti = rng.choice(target_choices)
        if ti == len(cand):
            cand.append([c])
        else:
            ipos = rng.randrange(len(cand[ti]) + 1)
            cand[ti].insert(ipos, c)
        cand = [r for r in cand if r]
    else:  # swap
        if sum(len(r) for r in cand) < 2:
            return None
        a = rng.randrange(len(cand)); b = rng.randrange(len(cand))
        if not cand[a] or not cand[b]:
            return None
        ia = rng.randrange(len(cand[a])); ib = rng.randrange(len(cand[b]))
        cand[a][ia], cand[b][ib] = cand[b][ib], cand[a][ia]

    if all(capacity_feasible(r, inst, cfg) for r in cand):
        return cand
    return None


def local_search(routes: List[List[int]], inst: Instance, cfg: S4Config, rng: random.Random) -> List[List[int]]:
    best = [r[:] for r in routes]
    best_obj = evaluate_solution(best, inst, cfg)["objective"]
    for _ in range(cfg.local_search_trials):
        cand = _random_local_move(best, inst, cfg, rng)
        if cand is None:
            continue
        obj = evaluate_solution(cand, inst, cfg)["objective"]
        if obj + EPS < best_obj:
            best, best_obj = cand, obj
    return best


def optimize_s4(inst: Instance, cfg: S4Config, seed: int = 42) -> Tuple[List[List[int]], Dict, pd.DataFrame]:
    rng = random.Random(seed)
    start = construct_initial_solution(inst, cfg)
    current = local_search(start, inst, cfg, rng)
    current_metrics = evaluate_solution(current, inst, cfg)
    best = copy.deepcopy(current)
    best_metrics = current_metrics.copy()

    destroy_ops = ["random", "worst_lateness", "related", "route"]
    repair_ops = ["greedy", "regret2", "urgency"]
    d_weights = {op: 1.0 for op in destroy_ops}
    r_weights = {op: 1.0 for op in repair_ops}
    d_scores = {op: 0.0 for op in destroy_ops}; d_uses = {op: 0 for op in destroy_ops}
    r_scores = {op: 0.0 for op in repair_ops}; r_uses = {op: 0 for op in repair_ops}

    temp = max(1.0, cfg.initial_temperature_ratio * current_metrics["objective"])
    history = []
    n = inst.n_customers
    no_improve = 0

    def weighted_choice(options, weights):
        vals = [weights[o] for o in options]
        return rng.choices(options, weights=vals, k=1)[0]

    for it in range(1, cfg.iterations + 1):
        dop = weighted_choice(destroy_ops, d_weights)
        rop = weighted_choice(repair_ops, r_weights)
        frac = rng.uniform(cfg.destroy_min_frac, cfg.destroy_max_frac)
        q = max(2, int(round(frac * n)))
        partial, removed = destroy_solution(current, inst, cfg, q, dop, rng)
        try:
            candidate = repair_solution(partial, removed, inst, cfg, rop, rng)
        except RuntimeError:
            continue
        if it % 3 == 0:
            candidate = local_search(candidate, inst, cfg, rng)
        cand_metrics = evaluate_solution(candidate, inst, cfg)
        delta = cand_metrics["objective"] - current_metrics["objective"]
        accepted = delta <= 0 or rng.random() < math.exp(-delta / max(temp, EPS))
        reward = 0.0
        if accepted:
            current, current_metrics = candidate, cand_metrics
            reward = 1.0
            if cand_metrics["objective"] + EPS < best_metrics["objective"]:
                best, best_metrics = copy.deepcopy(candidate), cand_metrics.copy()
                reward = 5.0
                no_improve = 0
            elif delta < 0:
                reward = 3.0
                no_improve += 1
            else:
                no_improve += 1
        else:
            no_improve += 1
        d_scores[dop] += reward; d_uses[dop] += 1
        r_scores[rop] += reward; r_uses[rop] += 1
        temp *= cfg.cooling_rate

        if it % cfg.segment_length == 0:
            for op in destroy_ops:
                perf = d_scores[op] / max(1, d_uses[op])
                d_weights[op] = (1-cfg.reaction_factor)*d_weights[op] + cfg.reaction_factor*max(0.1, perf)
                d_scores[op] = 0; d_uses[op] = 0
            for op in repair_ops:
                perf = r_scores[op] / max(1, r_uses[op])
                r_weights[op] = (1-cfg.reaction_factor)*r_weights[op] + cfg.reaction_factor*max(0.1, perf)
                r_scores[op] = 0; r_uses[op] = 0

        history.append({
            "iteration": it, "current_objective": current_metrics["objective"],
            "best_objective": best_metrics["objective"], "temperature": temp,
            "destroy_operator": dop, "repair_operator": rop, "accepted": accepted,
        })
        if (it >= cfg.early_stop_min_iterations and
                no_improve >= cfg.early_stop_patience):
            break

    best = local_search(best, inst, cfg, rng)
    best_metrics = evaluate_solution(best, inst, cfg)
    return best, best_metrics, pd.DataFrame(history)


def routes_to_frame(routes: List[List[int]], inst: Instance, cfg: S4Config, seed: int) -> pd.DataFrame:
    rows = []
    for v, route in enumerate(routes, start=1):
        m = evaluate_route(route, inst, cfg, mode="adaptive")
        rows.append({
            "delivery_date": inst.delivery_date, "vehicle_id": f"V{v}", "sequence": 0,
            "node_id": DEPOT_ID, "arrival_min": cfg.depot_start_min,
            "departure_min": cfg.depot_start_min, "lateness_min": 0.0,
            "quantile": "start", "seed": seed,
        })
        for seq, d in enumerate(m["details"], start=1):
            rows.append({
                "delivery_date": inst.delivery_date, "vehicle_id": f"V{v}", "sequence": seq,
                "node_id": inst.node_ids[d["node"]], "arrival_min": d["arrival_min"],
                "departure_min": d["departure_min"], "lateness_min": d["lateness_min"],
                "quantile": d["quantile"], "seed": seed,
            })
        rows.append({
            "delivery_date": inst.delivery_date, "vehicle_id": f"V{v}", "sequence": len(route)+1,
            "node_id": DEPOT_ID, "arrival_min": m["route_end_min"],
            "departure_min": m["route_end_min"], "lateness_min": 0.0,
            "quantile": "return", "seed": seed,
        })
    return pd.DataFrame(rows)


def s4_summary_row(inst: Instance, metrics: Dict, seed: int, runtime_sec: float) -> Dict:
    return {
        "delivery_date": inst.delivery_date,
        "seed": seed,
        "customer_count": inst.n_customers,
        "vehicle_count": metrics["vehicle_count"],
        "total_distance_km": metrics["distance_km"],
        "total_travel_time_min": metrics["travel_time_min"],
        "total_lateness_min": metrics["lateness_min"],
        "otd_pct": metrics["otd_pct"],
        "risk_penalty_min": metrics["risk_penalty_min"],
        "overtime_min": metrics["overtime_min"],
        "objective": metrics["objective"],
        "runtime_sec": runtime_sec,
    }


def run_s4_dates(repo: DataRepository, dates: Sequence[str], cfg: S4Config, seeds: Sequence[int], checkpoint_dir: str | Path) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, Dict[str, List[List[int]]]]:
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    route_frames = []
    summary_rows = []
    history_frames = []
    best_routes_by_date = {}

    for d_idx, date in enumerate(dates, start=1):
        inst = repo.get_instance(date)
        best_pack = None
        print(f"[S4 {d_idx}/{len(dates)}] {date}: {inst.n_customers} customers")
        for seed in seeds:
            t0 = time.perf_counter()
            routes, metrics, hist = optimize_s4(inst, cfg, seed=seed)
            runtime = time.perf_counter() - t0
            row = s4_summary_row(inst, metrics, seed, runtime)
            summary_rows.append(row)
            hist["delivery_date"] = date; hist["seed"] = seed
            history_frames.append(hist)
            if best_pack is None or metrics["objective"] < best_pack[1]["objective"]:
                best_pack = (routes, metrics, seed, runtime)
        routes, metrics, seed, runtime = best_pack
        best_routes_by_date[date] = routes
        route_frames.append(routes_to_frame(routes, inst, cfg, seed))
        # Per-date checkpoint.
        route_frames[-1].to_csv(checkpoint_dir / f"{date}_s4_routes.csv", index=False)
        pd.DataFrame([s4_summary_row(inst, metrics, seed, runtime)]).to_csv(checkpoint_dir / f"{date}_s4_best_summary.csv", index=False)

    all_runs = pd.DataFrame(summary_rows)
    best_summary = (all_runs.sort_values(["delivery_date", "objective"])
                    .groupby("delivery_date", as_index=False).first())
    routes_df = pd.concat(route_frames, ignore_index=True) if route_frames else pd.DataFrame()
    history_df = (pd.concat(history_frames, ignore_index=True) if history_frames else
                  pd.DataFrame(columns=["iteration", "current_objective", "best_objective",
                                        "temperature", "destroy_operator", "repair_operator",
                                        "accepted", "scenario", "delivery_date", "seed"]))
    return routes_df, best_summary, all_runs, history_df, best_routes_by_date


# ----------------------------- S5 ---------------------------------

def _shock_matrix(inst: Instance, shock: str) -> Tuple[np.ndarray, float]:
    if shock == "p85":
        return inst.tt85, 0.0
    if shock == "p95":
        return inst.tt95, 0.0
    if shock.startswith("delay"):
        return inst.tt50, float(shock.replace("delay", ""))
    raise ValueError(f"Unknown shock: {shock}")


def prefix_state(route: Sequence[int], served_count: int, inst: Instance, cfg: S5Config) -> Dict:
    t = cfg.depot_start_min
    prev = 0
    metrics = {"distance_km": 0.0, "travel_time_min": 0.0, "lateness_min": 0.0,
               "on_time_count": 0, "customer_count": 0, "waiting_min": 0.0}
    for node in list(route)[:served_count]:
        tt = inst.tt50[prev, node]
        metrics["distance_km"] += inst.dist[prev, node]
        metrics["travel_time_min"] += tt
        raw = t + tt
        wait = max(0.0, inst.tw_start[node] - raw)
        arr = raw + wait
        late = max(0.0, arr - inst.tw_end[node])
        metrics["waiting_min"] += wait
        metrics["lateness_min"] += late
        metrics["on_time_count"] += int(late <= EPS)
        metrics["customer_count"] += 1
        t = arr + inst.service[node]
        prev = node
    return {"time": t, "node": prev, "metrics": metrics}


def evaluate_continuation(
    order: Sequence[int], inst: Instance, shock: str, cfg: S5Config,
    start_time: float, start_node: int,
) -> Dict:
    matrix, one_time_delay = _shock_matrix(inst, shock)
    t = start_time + one_time_delay
    prev = start_node
    distance = 0.0; travel = 0.0; late_total = 0.0; wait_total = 0.0; on_time = 0
    details = []
    for node in order:
        tt = matrix[prev, node]
        distance += inst.dist[prev, node]; travel += tt
        raw = t + tt
        wait = max(0.0, inst.tw_start[node] - raw)
        arr = raw + wait
        late = max(0.0, arr - inst.tw_end[node])
        wait_total += wait; late_total += late; on_time += int(late <= EPS)
        t = arr + inst.service[node]
        details.append({"node": int(node), "arrival_min": arr, "lateness_min": late})
        prev = node
    tt_back = matrix[prev, 0]
    distance += inst.dist[prev, 0]; travel += tt_back; t += tt_back
    overtime = max(0.0, t - cfg.route_end_target_min)
    return {
        "distance_km": distance, "travel_time_min": travel, "lateness_min": late_total,
        "waiting_min": wait_total, "on_time_count": on_time, "customer_count": len(order),
        "route_end_min": t, "overtime_min": overtime, "details": details,
    }


def stability_metrics(original: Sequence[int], updated: Sequence[int], current_node: int = 0) -> Dict:
    original = list(original); updated = list(updated)
    m = len(original)
    if m <= 1:
        return {"position_shift_norm": 0.0, "edge_retention": 1.0, "route_stability_index": 1.0, "moved_stops": 0}
    pos_o = {c: i for i, c in enumerate(original)}
    pos_u = {c: i for i, c in enumerate(updated)}
    shift = sum(abs(pos_o[c] - pos_u[c]) for c in original) / (m * max(1, m-1))
    edges_o = set(zip([current_node] + original, original + [0]))
    edges_u = set(zip([current_node] + updated, updated + [0]))
    retain = len(edges_o & edges_u) / max(1, len(edges_o))
    rsi = 0.5 * (1.0 - shift) + 0.5 * retain
    moved = sum(pos_o[c] != pos_u[c] for c in original)
    return {"position_shift_norm": shift, "edge_retention": retain, "route_stability_index": rsi, "moved_stops": moved}


def continuation_objective(
    order: Sequence[int], original: Sequence[int], inst: Instance, shock: str, cfg: S5Config,
    start_time: float, start_node: int, risk_guided: bool, stability_aware: bool,
) -> Tuple[float, Dict, Dict]:
    m = evaluate_continuation(order, inst, shock, cfg, start_time, start_node)
    obj = (cfg.w_distance*m["distance_km"] + cfg.w_travel*m["travel_time_min"]
           + cfg.w_lateness*m["lateness_min"] + cfg.w_overtime*m["overtime_min"])
    risk_value = 0.0
    if risk_guided and len(order):
        p85 = evaluate_continuation(order, inst, "p85", cfg, start_time, start_node)
        p95 = evaluate_continuation(order, inst, "p95", cfg, start_time, start_node)
        risk_value = 0.4*p85["lateness_min"] + 0.6*p95["lateness_min"]
        obj += cfg.w_risk * risk_value
    sm = stability_metrics(original, order, current_node=start_node)
    if stability_aware:
        instability = 1.0 - sm["route_stability_index"]
        obj += cfg.stability_weight * instability
    return obj, m, {**sm, "risk_value": risk_value}


def distance_priority_order(remaining: Sequence[int], inst: Instance, current_node: int) -> List[int]:
    pool = list(remaining); out = []; cur = current_node
    while pool:
        nxt = min(pool, key=lambda c: (inst.dist[cur, c], inst.tw_end[c]))
        out.append(nxt); pool.remove(nxt); cur = nxt
    return out


def risk_priority_order(remaining: Sequence[int], inst: Instance, current_node: int, start_time: float) -> List[int]:
    pool = list(remaining); out = []; cur = current_node; t = start_time
    while pool:
        def score(c):
            arr50 = t + inst.tt50[cur, c]
            late50 = max(0.0, arr50 - inst.tw_end[c])
            extra95 = max(0.0, inst.tt95[cur, c] - inst.tt50[cur, c])
            slack = max(0.0, inst.tw_end[c] - arr50)
            return 0.25*inst.dist[cur, c] + 6.0*late50 + 0.5*extra95 + 0.002*slack
        nxt = min(pool, key=score)
        out.append(nxt); pool.remove(nxt)
        raw = t + inst.tt50[cur, nxt]
        t = max(raw, inst.tw_start[nxt]) + inst.service[nxt]
        cur = nxt
    return out


def _mutate_order(order: List[int], rng: random.Random) -> List[int]:
    n = len(order)
    if n < 2:
        return order[:]
    cand = order[:]
    move = rng.choice(["swap", "relocate", "2opt"])
    if move == "swap":
        i, j = rng.sample(range(n), 2)
        cand[i], cand[j] = cand[j], cand[i]
    elif move == "relocate":
        i, j = rng.sample(range(n), 2)
        x = cand.pop(i); cand.insert(j, x)
    else:
        i, j = sorted(rng.sample(range(n), 2))
        cand[i:j+1] = reversed(cand[i:j+1])
    return cand


def optimize_remaining(
    original: Sequence[int], inst: Instance, shock: str, cfg: S5Config,
    start_time: float, start_node: int, method: str, seed: int,
) -> Tuple[List[int], Dict, Dict, float]:
    original = list(original)
    if len(original) <= 1:
        obj, m, sm = continuation_objective(original, original, inst, shock, cfg, start_time, start_node,
                                             risk_guided=method in {"RG", "SA"}, stability_aware=method == "SA")
        return original, m, sm, obj
    if method == "NR":
        init = original
        iterations = 0; risk = False; stability = False
    elif method == "DP":
        init = distance_priority_order(original, inst, start_node)
        iterations = 0; risk = False; stability = False
    elif method == "FR":
        init = min([original, distance_priority_order(original, inst, start_node)],
                   key=lambda o: continuation_objective(o, original, inst, shock, cfg, start_time, start_node, False, False)[0])
        iterations = cfg.iterations_fr; risk = False; stability = False
    elif method == "RG":
        # Include the unchanged continuation as a candidate. This prevents the
        # risk-guided benchmark from being forced to adopt a worse re-sequencing.
        candidates = [original, risk_priority_order(original, inst, start_node, start_time)]
        init = min(candidates, key=lambda o: continuation_objective(
            o, original, inst, shock, cfg, start_time, start_node, True, False
        )[0])
        iterations = cfg.iterations_rg; risk = True; stability = False
    elif method == "SA":
        candidates = [original, risk_priority_order(original, inst, start_node, start_time)]
        init = min(candidates, key=lambda o: continuation_objective(o, original, inst, shock, cfg, start_time, start_node, True, True)[0])
        iterations = cfg.iterations_sa; risk = True; stability = True
    else:
        raise ValueError(method)

    rng = random.Random(seed)
    current = init[:]
    cur_obj, cur_m, cur_sm = continuation_objective(current, original, inst, shock, cfg, start_time, start_node, risk, stability)
    best = current[:]; best_obj = cur_obj; best_m = cur_m; best_sm = cur_sm
    temp = max(1.0, cfg.initial_temperature_ratio * cur_obj)
    no_improve = 0
    for it in range(1, iterations + 1):
        cand = _mutate_order(current, rng)
        obj, m, sm = continuation_objective(cand, original, inst, shock, cfg, start_time, start_node, risk, stability)
        delta = obj - cur_obj
        if delta <= 0 or rng.random() < math.exp(-delta / max(temp, EPS)):
            current, cur_obj, cur_m, cur_sm = cand, obj, m, sm
        if obj + EPS < best_obj:
            best, best_obj, best_m, best_sm = cand, obj, m, sm
            no_improve = 0
        else:
            no_improve += 1
        temp *= cfg.cooling_rate
        if (it >= cfg.early_stop_min_iterations and
                no_improve >= cfg.early_stop_patience):
            break
    return best, best_m, best_sm, best_obj


def run_s5(
    repo: DataRepository, dates: Sequence[str], best_routes_by_date: Dict[str, List[List[int]]],
    cfg: S5Config, seeds: Sequence[int], checkpoint_dir: str | Path,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    checkpoint_dir = Path(checkpoint_dir); checkpoint_dir.mkdir(parents=True, exist_ok=True)
    rows = []; sequence_rows = []
    total_jobs = len(dates)*len(cfg.trigger_fractions)*len(cfg.shocks)*len(seeds)
    job = 0
    for date in dates:
        result_ck = checkpoint_dir / f"{date}_s5_results.csv"
        seq_ck = checkpoint_dir / f"{date}_s5_sequences.csv"
        expected_rows = len(cfg.trigger_fractions) * len(cfg.shocks) * len(seeds) * len(cfg.methods)
        if result_ck.exists() and seq_ck.exists():
            cached_r = pd.read_csv(result_ck)
            cached_s = pd.read_csv(seq_ck)
            if len(cached_r) == expected_rows:
                print(f"[S5 resume] {date}")
                rows.extend(cached_r.to_dict("records"))
                sequence_rows.extend(cached_s.to_dict("records"))
                continue
        inst = repo.get_instance(date)
        routes = best_routes_by_date[date]
        for trigger in cfg.trigger_fractions:
            for shock in cfg.shocks:
                for seed in seeds:
                    job += 1
                    print(f"[S5 {job}/{total_jobs}] {date} trigger={trigger:.2f} shock={shock} seed={seed}")
                    method_agg = {m: {
                        "distance_km": 0.0, "travel_time_min": 0.0, "lateness_min": 0.0,
                        "waiting_min": 0.0, "on_time_count": 0, "customer_count": 0,
                        "overtime_min": 0.0, "rsi_weighted": 0.0, "edge_weighted": 0.0,
                        "shift_weighted": 0.0, "moved_stops": 0, "remaining_stops": 0,
                        "runtime_sec": 0.0,
                    } for m in cfg.methods}

                    for v_idx, route in enumerate(routes, start=1):
                        if not route:
                            continue
                        served_count = min(len(route), max(0, int(math.floor(trigger*len(route)))))
                        # Keep at least one remaining stop when possible.
                        if len(route) > 1:
                            served_count = min(served_count, len(route)-1)
                        prefix = route[:served_count]
                        remaining = route[served_count:]
                        state = prefix_state(route, served_count, inst, cfg)
                        prefix_m = state["metrics"]
                        for method in cfg.methods:
                            t0 = time.perf_counter()
                            updated, cont_m, sm, obj = optimize_remaining(
                                remaining, inst, shock, cfg, state["time"], state["node"],
                                method, seed + 1009*v_idx,
                            )
                            runtime = time.perf_counter() - t0
                            a = method_agg[method]
                            a["distance_km"] += prefix_m["distance_km"] + cont_m["distance_km"]
                            a["travel_time_min"] += prefix_m["travel_time_min"] + cont_m["travel_time_min"]
                            a["lateness_min"] += prefix_m["lateness_min"] + cont_m["lateness_min"]
                            a["waiting_min"] += prefix_m["waiting_min"] + cont_m["waiting_min"]
                            a["on_time_count"] += prefix_m["on_time_count"] + cont_m["on_time_count"]
                            a["customer_count"] += prefix_m["customer_count"] + cont_m["customer_count"]
                            a["overtime_min"] += cont_m["overtime_min"]
                            weight = len(remaining)
                            a["rsi_weighted"] += sm["route_stability_index"]*weight
                            a["edge_weighted"] += sm["edge_retention"]*weight
                            a["shift_weighted"] += sm["position_shift_norm"]*weight
                            a["moved_stops"] += sm["moved_stops"]
                            a["remaining_stops"] += weight
                            a["runtime_sec"] += runtime
                            for seq, node in enumerate(prefix + updated):
                                sequence_rows.append({
                                    "delivery_date": date, "trigger_fraction": trigger, "shock": shock,
                                    "seed": seed, "method": method, "vehicle_id": f"V{v_idx}",
                                    "sequence": seq+1, "node_id": inst.node_ids[node],
                                    "served_frozen": seq < served_count,
                                })

                    for method, a in method_agg.items():
                        rem = max(1, a["remaining_stops"])
                        rows.append({
                            "delivery_date": date, "trigger_fraction": trigger, "shock": shock,
                            "seed": seed, "method": method,
                            "total_distance_km": a["distance_km"],
                            "total_travel_time_min": a["travel_time_min"],
                            "total_lateness_min": a["lateness_min"],
                            "otd_pct": 100*a["on_time_count"]/max(1,a["customer_count"]),
                            "overtime_min": a["overtime_min"],
                            "route_stability_index": a["rsi_weighted"]/rem,
                            "edge_retention": a["edge_weighted"]/rem,
                            "position_shift_norm": a["shift_weighted"]/rem,
                            "moved_stops": a["moved_stops"],
                            "remaining_stops": a["remaining_stops"],
                            "runtime_sec": a["runtime_sec"],
                        })
        # checkpoint after each completed date
        date_rows = pd.DataFrame([r for r in rows if r["delivery_date"] == date])
        date_seq = pd.DataFrame([r for r in sequence_rows if r["delivery_date"] == date])
        date_rows.to_csv(result_ck, index=False)
        date_seq.to_csv(seq_ck, index=False)

    result = pd.DataFrame(rows)
    seq_df = pd.DataFrame(sequence_rows)
    keys = ["delivery_date", "trigger_fraction", "shock", "seed"]
    nr = result[result["method"] == "NR"][keys + ["total_lateness_min", "otd_pct", "total_distance_km", "total_travel_time_min"]].copy()
    nr = nr.rename(columns={
        "total_lateness_min": "nr_lateness_min", "otd_pct": "nr_otd_pct",
        "total_distance_km": "nr_distance_km", "total_travel_time_min": "nr_travel_time_min",
    })
    result = result.merge(nr, on=keys, how="left")
    # Absolute lateness reduction is always defined. Percentage gain is only
    # meaningful when the no-replanning baseline has positive lateness.
    result["lateness_reduction_min"] = result["nr_lateness_min"] - result["total_lateness_min"]
    positive_nr = result["nr_lateness_min"] > EPS
    result["replanning_gain_lateness_pct"] = np.where(
        positive_nr,
        100.0 * result["lateness_reduction_min"] / result["nr_lateness_min"],
        np.nan,
    )
    result["replanning_avoids_new_lateness"] = np.where(
        positive_nr,
        result["total_lateness_min"] <= result["nr_lateness_min"] + EPS,
        result["total_lateness_min"] <= EPS,
    )
    result["otd_improvement_pp"] = result["otd_pct"]-result["nr_otd_pct"]
    result["distance_premium_pct"] = 100*(result["total_distance_km"]-result["nr_distance_km"])/result["nr_distance_km"].clip(lower=EPS)
    result["travel_time_premium_pct"] = 100*(result["total_travel_time_min"]-result["nr_travel_time_min"])/result["nr_travel_time_min"].clip(lower=EPS)
    return result, seq_df


def summarize_s5(results: pd.DataFrame) -> pd.DataFrame:
    return (results.groupby(["method", "shock", "trigger_fraction"], as_index=False)
            .agg(
                n_blocks=("delivery_date", "size"),
                mean_lateness_min=("total_lateness_min", "mean"),
                median_lateness_min=("total_lateness_min", "median"),
                mean_otd_pct=("otd_pct", "mean"),
                mean_lateness_reduction_min=("lateness_reduction_min", "mean"),
                mean_replanning_gain_pct=("replanning_gain_lateness_pct", "mean"),
                replanning_no_harm_rate_pct=("replanning_avoids_new_lateness", lambda s: 100.0*float(np.mean(s))),
                mean_otd_improvement_pp=("otd_improvement_pp", "mean"),
                mean_distance_premium_pct=("distance_premium_pct", "mean"),
                mean_travel_time_premium_pct=("travel_time_premium_pct", "mean"),
                mean_route_stability_index=("route_stability_index", "mean"),
                mean_edge_retention=("edge_retention", "mean"),
                mean_moved_stops=("moved_stops", "mean"),
                mean_runtime_sec=("runtime_sec", "mean"),
            ))


def statistical_tests(results: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    try:
        from scipy.stats import friedmanchisquare, wilcoxon
    except Exception as e:
        return pd.DataFrame([{"test": "unavailable", "error": str(e)}]), pd.DataFrame()

    block_cols = ["delivery_date", "trigger_fraction", "shock", "seed"]
    pivot = results.pivot_table(index=block_cols, columns="method", values="total_lateness_min", aggfunc="first").dropna()
    omnibus_rows = []
    if pivot.shape[0] >= 2 and pivot.shape[1] >= 3:
        try:
            stat, p = friedmanchisquare(*[pivot[c].values for c in pivot.columns])
            omnibus_rows.append({"test": "Friedman", "metric": "total_lateness_min", "statistic": stat, "p_value": p, "n_blocks": len(pivot)})
        except Exception as e:
            omnibus_rows.append({"test": "Friedman", "metric": "total_lateness_min", "error": str(e)})

    pair_rows = []
    comparisons = [("SA", m) for m in ["NR", "DP", "FR", "RG"] if m in pivot.columns and "SA" in pivot.columns]
    raw_ps = []
    for a, b in comparisons:
        x = pivot[a].values; y = pivot[b].values
        diff = x-y
        try:
            stat, p = wilcoxon(x, y, zero_method="wilcox", alternative="two-sided")
        except Exception:
            stat, p = np.nan, 1.0
        nonzero = diff[np.abs(diff) > EPS]
        rank_biserial = np.nan
        if len(nonzero):
            # Probability-of-superiority style signed effect; negative means A has lower lateness.
            rank_biserial = (np.sum(nonzero > 0) - np.sum(nonzero < 0)) / len(nonzero)
        pair_rows.append({"method_a": a, "method_b": b, "statistic": stat, "p_value_raw": p,
                          "median_difference_a_minus_b": float(np.median(diff)),
                          "signed_effect": rank_biserial, "n_blocks": len(diff)})
        raw_ps.append(p)
    # Holm correction.
    if pair_rows:
        order = np.argsort(raw_ps)
        adjusted = np.empty(len(raw_ps), dtype=float)
        running = 0.0
        m = len(raw_ps)
        for rank, idx in enumerate(order):
            val = min(1.0, (m-rank)*raw_ps[idx])
            running = max(running, val)
            adjusted[idx] = running
        for row, adj in zip(pair_rows, adjusted):
            row["p_value_holm"] = adj
            row["significant_0_05"] = bool(adj < 0.05)
    return pd.DataFrame(omnibus_rows), pd.DataFrame(pair_rows)


def export_results(
    output_dir: str | Path, repo: DataRepository, s4_routes: pd.DataFrame,
    s4_best: pd.DataFrame, s4_all_runs: pd.DataFrame, s4_history: pd.DataFrame,
    s5_results: pd.DataFrame, s5_sequences: pd.DataFrame, config_payload: Dict,
) -> Path:
    output_dir = Path(output_dir); output_dir.mkdir(parents=True, exist_ok=True)
    audit = repo.audit()
    s5_summary = summarize_s5(s5_results)
    omnibus, pairwise = statistical_tests(s5_results)

    files = {
        "input_validation_report.csv": audit,
        "s4_routes.csv": s4_routes,
        "s4_best_summary.csv": s4_best,
        "s4_all_seed_runs.csv": s4_all_runs,
        "s4_iteration_history.csv": s4_history,
        "s5_results_detail.csv": s5_results,
        "s5_updated_sequences.csv": s5_sequences,
        "s5_summary.csv": s5_summary,
        "s5_friedman_test.csv": omnibus,
        "s5_pairwise_wilcoxon_holm.csv": pairwise,
    }
    for fn, df in files.items():
        df.to_csv(output_dir/fn, index=False)

    with open(output_dir/"experiment_config.json", "w", encoding="utf-8") as f:
        json.dump(config_payload, f, ensure_ascii=False, indent=2, default=str)

    # Excel workbook.
    xlsx = output_dir/"S4_S5_Soft_Computing_Results.xlsx"
    with pd.ExcelWriter(xlsx, engine="openpyxl") as writer:
        for sheet, df in [
            ("Input_Audit", audit), ("S4_Best", s4_best), ("S4_All_Runs", s4_all_runs),
            ("S4_Routes", s4_routes), ("S5_Detail", s5_results), ("S5_Summary", s5_summary),
            ("Friedman", omnibus), ("Pairwise", pairwise),
        ]:
            df.to_excel(writer, sheet_name=sheet[:31], index=False)

    # Figures are optional and created only when matplotlib is available.
    try:
        import matplotlib.pyplot as plt
        figdir = output_dir/"figures"; figdir.mkdir(exist_ok=True)
        agg = s5_results.groupby("method", as_index=False).agg(
            lateness=("total_lateness_min", "mean"), otd=("otd_pct", "mean"),
            stability=("route_stability_index", "mean"), runtime=("runtime_sec", "mean"),
        )
        for col, ylabel, fn in [
            ("lateness", "Mean total lateness (min)", "mean_lateness_by_method.png"),
            ("otd", "Mean OTD (%)", "mean_otd_by_method.png"),
            ("stability", "Mean route stability index", "mean_stability_by_method.png"),
            ("runtime", "Mean runtime (s)", "mean_runtime_by_method.png"),
        ]:
            fig, ax = plt.subplots(figsize=(8,5))
            ax.bar(agg["method"], agg[col])
            ax.set_xlabel("Method"); ax.set_ylabel(ylabel); ax.grid(axis="y", alpha=.25)
            fig.tight_layout(); fig.savefig(figdir/fn, dpi=300); plt.close(fig)

        trade = s5_results.groupby("method", as_index=False).agg(
            gain=("replanning_gain_lateness_pct", "mean"),
            stability=("route_stability_index", "mean"),
        )
        fig, ax = plt.subplots(figsize=(7,5))
        ax.scatter(trade["gain"], trade["stability"])
        for _, r in trade.iterrows():
            ax.annotate(r["method"], (r["gain"], r["stability"]))
        ax.set_xlabel("Mean lateness re-planning gain (%)")
        ax.set_ylabel("Mean route stability index")
        ax.grid(alpha=.25); fig.tight_layout(); fig.savefig(figdir/"gain_stability_tradeoff.png", dpi=300); plt.close(fig)
    except Exception as e:
        with open(output_dir/"figure_generation_error.txt", "w") as f:
            f.write(str(e))

    zip_path = output_dir.parent/(output_dir.name + ".zip")
    if zip_path.exists():
        zip_path.unlink()
    shutil.make_archive(str(zip_path.with_suffix("")), "zip", root_dir=output_dir)
    return zip_path

# ===================== END-TO-END RAW-DATA + S0-S5 EXTENSIONS =====================


def _time_value_to_minutes(value, default=None):
    """Convert Excel/pandas time-like values to minutes after midnight."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return default
    if hasattr(value, "hour") and hasattr(value, "minute"):
        return float(value.hour * 60 + value.minute + getattr(value, "second", 0) / 60.0)
    if isinstance(value, (int, float)):
        # Excel time fraction or already minutes.
        v = float(value)
        if 0 <= v < 1:
            return v * 1440.0
        if 0 <= v <= 1440:
            return v
    text = str(value).strip()
    if not text or text.lower() in {"nan", "nat", "none"}:
        return default
    try:
        parts = text.split(":")
        if len(parts) >= 2:
            return float(int(parts[0]) * 60 + int(parts[1]) + (float(parts[2]) / 60.0 if len(parts) > 2 else 0.0))
    except Exception:
        pass
    parsed = pd.to_datetime(text, errors="coerce")
    if pd.isna(parsed):
        return default
    return float(parsed.hour * 60 + parsed.minute + parsed.second / 60.0)


def prepare_normalized_input_from_raw(
    raw_xlsx_path: str | Path,
    matrix_zip_path: str | Path,
    workdir: str | Path,
    stops_sheet: str = "Stops",
) -> Tuple[Path, pd.DataFrame, pd.DataFrame]:
    """
    Convert the operational Excel workbook to customer-day stops and combine it
    with the already validated OSRM–TomTom saved matrices.

    Returns
    -------
    normalized_dir, raw_audit, coordinate_audit
    """
    raw_xlsx_path = Path(raw_xlsx_path)
    matrix_zip_path = Path(matrix_zip_path)
    workdir = Path(workdir)
    normalized = workdir / "normalized_input"
    extracted = workdir / "matrix_extracted"
    if normalized.exists():
        shutil.rmtree(normalized)
    if extracted.exists():
        shutil.rmtree(extracted)
    normalized.mkdir(parents=True, exist_ok=True)
    extracted.mkdir(parents=True, exist_ok=True)

    raw = pd.read_excel(raw_xlsx_path, sheet_name=stops_sheet)
    required = {
        "order_number", "customer_code", "customer_name", "longitude", "latitude",
        "total_volume", "total_weight", "first_delivery_dt", "tw_start", "tw_end",
        "service_time_min_baseline",
    }
    missing = required - set(raw.columns)
    if missing:
        raise ValueError(f"Raw workbook sheet '{stops_sheet}' is missing columns: {sorted(missing)}")

    raw = raw.copy()
    raw["first_delivery_dt"] = pd.to_datetime(raw["first_delivery_dt"], errors="coerce")
    if raw["first_delivery_dt"].isna().any():
        bad = raw.loc[raw["first_delivery_dt"].isna(), ["order_number", "first_delivery_dt"]].head()
        raise ValueError(f"Unparseable first_delivery_dt values:\n{bad}")
    raw["delivery_date"] = raw["first_delivery_dt"].dt.strftime("%Y-%m-%d")
    raw["customer_code"] = raw["customer_code"].astype(str).str.strip()
    raw["customer_name"] = raw["customer_name"].astype(str).str.strip()
    for c in ["longitude", "latitude", "total_volume", "total_weight", "service_time_min_baseline"]:
        raw[c] = pd.to_numeric(raw[c], errors="coerce")
    if raw[["longitude", "latitude", "total_volume", "total_weight"]].isna().any().any():
        bad = raw[raw[["longitude", "latitude", "total_volume", "total_weight"]].isna().any(axis=1)].head()
        raise ValueError(f"Missing/invalid coordinate or demand values in raw data:\n{bad}")
    if (raw[["total_volume", "total_weight"]] < 0).any().any():
        raise ValueError("Negative weight or volume found in raw data.")

    raw["tw_start_min"] = raw["tw_start"].map(lambda x: _time_value_to_minutes(x, 480.0))
    raw["tw_end_min"] = raw["tw_end"].map(lambda x: _time_value_to_minutes(x, 900.0))
    raw["service_time_min"] = raw["service_time_min_baseline"].fillna(8.0).astype(float)

    # Coordinate consistency audit before aggregation.
    coordinate_audit = (raw.groupby(["delivery_date", "customer_code"], as_index=False)
                        .agg(order_count=("order_number", "size"),
                             lon_min=("longitude", "min"), lon_max=("longitude", "max"),
                             lat_min=("latitude", "min"), lat_max=("latitude", "max")))
    coordinate_audit["lon_spread"] = coordinate_audit["lon_max"] - coordinate_audit["lon_min"]
    coordinate_audit["lat_spread"] = coordinate_audit["lat_max"] - coordinate_audit["lat_min"]
    coordinate_audit["coordinate_consistent"] = (
        coordinate_audit["lon_spread"].abs() <= 1e-7
    ) & (coordinate_audit["lat_spread"].abs() <= 1e-7)

    aggregations = {
        "customer_name": "first",
        "longitude": "first",
        "latitude": "first",
        "order_number": "size",
        "total_weight": "sum",
        "total_volume": "sum",
        "tw_start_min": "max",   # all orders must be ready
        "tw_end_min": "min",     # earliest deadline governs the combined stop
        "service_time_min": "max",  # reproduces the matrix-builder preparation rule
    }
    if "quantity_order" in raw.columns:
        raw["quantity_order"] = pd.to_numeric(raw["quantity_order"], errors="coerce").fillna(0)
        aggregations["quantity_order"] = "sum"

    grouped = (raw.groupby(["delivery_date", "customer_code"], as_index=False)
               .agg(aggregations)
               .rename(columns={"order_number": "order_count"}))
    grouped["node_id"] = grouped["customer_code"].astype(str)
    grouped = grouped.rename(columns={
        "total_weight": "weight_kg",
        "total_volume": "volume_cm3",
    })
    grouped["tw_start_min"] = grouped["tw_start_min"].astype(float)
    grouped["tw_end_min"] = grouped["tw_end_min"].astype(float)
    grouped["service_time_min"] = grouped["service_time_min"].astype(float)
    grouped = grouped.sort_values(["delivery_date", "node_id"]).reset_index(drop=True)

    if grouped.duplicated(["delivery_date", "node_id"]).any():
        raise RuntimeError("Customer-day aggregation still contains duplicate date-node rows.")
    if (grouped["tw_end_min"] < grouped["tw_start_min"]).any():
        raise ValueError("At least one aggregated time window ends before it starts.")

    stops_cols = [
        "delivery_date", "node_id", "weight_kg", "volume_cm3",
        "tw_start_min", "tw_end_min", "service_time_min",
    ]
    grouped[stops_cols].to_csv(normalized / "stops.csv", index=False)
    grouped.to_csv(workdir / "customer_day_stops_full_audit.csv", index=False)
    coordinate_audit.to_csv(workdir / "coordinate_consistency_audit.csv", index=False)

    with zipfile.ZipFile(matrix_zip_path) as zf:
        required_matrix_files = [
            "combined_osrm_distance_matrix_km_long.csv",
            "combined_travel_time_matrix_p50_long.csv",
            "combined_travel_time_matrix_p85_long.csv",
            "combined_travel_time_matrix_p95_long.csv",
        ]
        names = set(zf.namelist())
        for fn in required_matrix_files:
            candidates = [n for n in names if Path(n).name == fn]
            if not candidates:
                raise FileNotFoundError(f"Matrix ZIP does not contain {fn}")
            src_name = sorted(candidates, key=len)[0]
            # Normalize value-column names so DataRepository receives a stable schema.
            with zf.open(src_name) as src:
                frame = pd.read_csv(src)
            if fn.startswith("combined_travel_time_matrix_"):
                value_candidates = [c for c in frame.columns if c.startswith("travel_time_min")]
                if len(value_candidates) != 1:
                    raise ValueError(f"Cannot identify travel-time value column in {fn}: {list(frame.columns)}")
                frame = frame.rename(columns={value_candidates[0]: "travel_time_min"})
            frame.to_csv(normalized / fn, index=False)
        # Keep selected audit inputs when available.
        for audit_name in [
            "matrix_source_audit.csv", "matrix_manifest.csv", "tomtom_traffic_multiplier.csv",
            "audit/matrix_coverage_summary.csv", "audit/customer_day_stops_for_matrix_builder.csv",
        ]:
            if audit_name in names:
                target = workdir / "matrix_audit" / Path(audit_name).name
                target.parent.mkdir(parents=True, exist_ok=True)
                with zf.open(audit_name) as src, open(target, "wb") as dst:
                    shutil.copyfileobj(src, dst)

    # High-level raw-data audit.
    split_parts = np.maximum(
        np.ceil(grouped["weight_kg"].to_numpy(float) / 10000.0),
        np.ceil(grouped["volume_cm3"].to_numpy(float) / 3992800.0),
    ).astype(int)
    raw_audit = pd.DataFrame([
        {"metric": "raw_order_rows", "value": int(len(raw))},
        {"metric": "unique_customers", "value": int(raw["customer_code"].nunique())},
        {"metric": "delivery_dates", "value": int(raw["delivery_date"].nunique())},
        {"metric": "customer_day_stops", "value": int(len(grouped))},
        {"metric": "date_min", "value": str(grouped["delivery_date"].min())},
        {"metric": "date_max", "value": str(grouped["delivery_date"].max())},
        {"metric": "missing_coordinates", "value": int(raw[["longitude", "latitude"]].isna().any(axis=1).sum())},
        {"metric": "coordinate_inconsistent_customer_days", "value": int((~coordinate_audit["coordinate_consistent"]).sum())},
        {"metric": "oversize_customer_day_stops_2d", "value": int(np.sum(split_parts > 1))},
        {"metric": "virtual_stops_after_split", "value": int(split_parts.sum())},
    ])
    raw_audit.to_csv(workdir / "raw_preprocessing_audit.csv", index=False)
    return normalized, raw_audit, coordinate_audit


SCENARIO_SPECS = {
    "S0": {"name": "VRPTW-1D Static Baseline", "matrix": "p50", "use_volume": False, "risk_guided": False},
    "S1": {"name": "MCVRPTW-TD p50 Reference", "matrix": "p50", "use_volume": True, "risk_guided": False},
    "S2": {"name": "Fixed p85", "matrix": "p85", "use_volume": True, "risk_guided": False},
    "S3": {"name": "Fixed p95", "matrix": "p95", "use_volume": True, "risk_guided": False},
    "S4": {"name": "QRG-ALNS-ILS Proposed", "matrix": "adaptive", "use_volume": True, "risk_guided": True},
}


def _copy_instance_for_scenario(inst: Instance, scenario_id: str) -> Instance:
    spec = SCENARIO_SPECS[scenario_id]
    if spec["matrix"] == "adaptive":
        tt50, tt85, tt95 = inst.tt50.copy(), inst.tt85.copy(), inst.tt95.copy()
    else:
        selected = {"p50": inst.tt50, "p85": inst.tt85, "p95": inst.tt95}[spec["matrix"]]
        # Equal matrices make the common ALNS backbone behave as a fixed-quantile model.
        tt50 = selected.copy(); tt85 = selected.copy(); tt95 = selected.copy()
    return Instance(
        delivery_date=inst.delivery_date,
        node_ids=list(inst.node_ids),
        weights=inst.weights.copy(),
        volumes=inst.volumes.copy(),
        tw_start=inst.tw_start.copy(),
        tw_end=inst.tw_end.copy(),
        service=inst.service.copy(),
        dist=inst.dist.copy(),
        tt50=tt50, tt85=tt85, tt95=tt95,
    )


def _config_for_scenario(base_cfg: S4Config, scenario_id: str) -> S4Config:
    cfg = copy.deepcopy(base_cfg)
    spec = SCENARIO_SPECS[scenario_id]
    if not spec["use_volume"]:
        cfg.capacity_volume_cm3 = 1e30
        cfg.w_capacity_imbalance = 0.0
    if not spec["risk_guided"]:
        cfg.w_risk = 0.0
    return cfg


def _actual_capacity_audit(routes: Sequence[Sequence[int]], inst: Instance,
                           weight_cap: float, volume_cap: float) -> Dict:
    route_weights = [float(inst.weights[np.asarray(r, dtype=int)].sum()) if r else 0.0 for r in routes]
    route_volumes = [float(inst.volumes[np.asarray(r, dtype=int)].sum()) if r else 0.0 for r in routes]
    return {
        "weight_violation_routes": int(sum(w > weight_cap + EPS for w in route_weights)),
        "volume_violation_routes": int(sum(v > volume_cap + EPS for v in route_volumes)),
        "max_weight_overload_kg": float(max([max(0.0, w-weight_cap) for w in route_weights] or [0.0])),
        "max_volume_overload_cm3": float(max([max(0.0, v-volume_cap) for v in route_volumes] or [0.0])),
        "mean_weight_util_pct": float(np.mean([100*w/weight_cap for w in route_weights])) if route_weights else 0.0,
        "mean_volume_util_pct": float(np.mean([100*v/volume_cap for v in route_volumes])) if route_volumes else 0.0,
        "max_weight_util_pct": float(max([100*w/weight_cap for w in route_weights] or [0.0])),
        "max_volume_util_pct": float(max([100*v/volume_cap for v in route_volumes] or [0.0])),
    }


def _scenario_route_frame(routes: List[List[int]], inst: Instance, cfg: S4Config,
                          scenario_id: str, seed: int) -> pd.DataFrame:
    spec = SCENARIO_SPECS[scenario_id]
    fixed_mode = "adaptive" if scenario_id == "S4" else "p50"
    rows = []
    for v, route in enumerate(routes, start=1):
        m = evaluate_route(route, inst, cfg, mode=fixed_mode)
        rows.append({
            "delivery_date": inst.delivery_date, "scenario": scenario_id,
            "scenario_name": spec["name"], "vehicle_id": f"V{v}", "sequence": 0,
            "node_id": DEPOT_ID, "arrival_min": cfg.depot_start_min,
            "departure_min": cfg.depot_start_min, "lateness_min": 0.0,
            "quantile": "start", "seed": seed,
        })
        for seq, d in enumerate(m["details"], start=1):
            q = d["quantile"] if scenario_id == "S4" else spec["matrix"]
            rows.append({
                "delivery_date": inst.delivery_date, "scenario": scenario_id,
                "scenario_name": spec["name"], "vehicle_id": f"V{v}", "sequence": seq,
                "node_id": inst.node_ids[d["node"]], "arrival_min": d["arrival_min"],
                "departure_min": d["departure_min"], "lateness_min": d["lateness_min"],
                "quantile": q, "seed": seed,
            })
        rows.append({
            "delivery_date": inst.delivery_date, "scenario": scenario_id,
            "scenario_name": spec["name"], "vehicle_id": f"V{v}", "sequence": len(route)+1,
            "node_id": DEPOT_ID, "arrival_min": m["route_end_min"],
            "departure_min": m["route_end_min"], "lateness_min": 0.0,
            "quantile": "return", "seed": seed,
        })
    return pd.DataFrame(rows)


def _routes_from_checkpoint(route_file: Path, inst: Instance) -> List[List[int]]:
    frame = pd.read_csv(route_file)
    node_to_idx = {str(n): i for i, n in enumerate(inst.node_ids)}
    routes = []
    for _, g in frame.sort_values(["vehicle_id", "sequence"]).groupby("vehicle_id", sort=False):
        ids = [str(x) for x in g["node_id"].tolist() if str(x) != DEPOT_ID]
        route = [node_to_idx[x] for x in ids if x in node_to_idx]
        if route:
            routes.append(route)
    return routes


def run_s0_s4_scenarios(
    repo: DataRepository,
    dates: Sequence[str],
    base_cfg: S4Config,
    seeds,
    checkpoint_dir: str | Path,
    scenarios: Sequence[str] = ("S0", "S1", "S2", "S3", "S4"),
    force_recompute: bool = False,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, Dict[str, Dict[str, List[List[int]]]]]:
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    route_frames = []
    all_rows = []
    history_frames = []
    best_routes: Dict[str, Dict[str, List[List[int]]]] = {s: {} for s in scenarios}

    for s_idx, scenario_id in enumerate(scenarios, start=1):
        spec = SCENARIO_SPECS[scenario_id]
        cfg = _config_for_scenario(base_cfg, scenario_id)
        seeds_here = list(seeds.get(scenario_id, [])) if isinstance(seeds, dict) else list(seeds)
        print(f"\n=== {scenario_id}: {spec['name']} ({s_idx}/{len(scenarios)}) — seeds={seeds_here} ===")
        for d_idx, date in enumerate(dates, start=1):
            base_inst = repo.get_instance(date)
            inst = _copy_instance_for_scenario(base_inst, scenario_id)
            best_pack = None
            print(f"[{scenario_id} {d_idx}/{len(dates)}] {date}: {inst.n_customers} virtual stops")
            for seed in seeds_here:
                route_ck = checkpoint_dir / f"{scenario_id}_{date}_seed{seed}_routes.csv"
                metrics_ck = checkpoint_dir / f"{scenario_id}_{date}_seed{seed}_metrics.json"
                if route_ck.exists() and metrics_ck.exists() and not force_recompute:
                    routes = _routes_from_checkpoint(route_ck, inst)
                    saved = json.loads(metrics_ck.read_text())
                    metrics = saved["metrics"]
                    audit = saved["audit"]
                    runtime = float(saved.get("runtime_sec", 0.0))
                    hist = pd.DataFrame()
                    print("  resumed seed", seed)
                else:
                    t0 = time.perf_counter()
                    routes, _, hist = optimize_s4(inst, cfg, seed=seed)
                    runtime = time.perf_counter() - t0
                    report_mode = "adaptive" if scenario_id == "S4" else "p50"
                    metrics = evaluate_solution(routes, inst, cfg, mode=report_mode)
                    audit = _actual_capacity_audit(
                        routes, base_inst,
                        weight_cap=base_cfg.capacity_weight_kg,
                        volume_cap=base_cfg.capacity_volume_cm3,
                    )
                    frame_seed = _scenario_route_frame(routes, inst, cfg, scenario_id, seed)
                    frame_seed.to_csv(route_ck, index=False)
                    metrics_ck.write_text(json.dumps({
                        "metrics": {k:v for k,v in metrics.items() if k != "route_metrics"},
                        "audit": audit, "runtime_sec": runtime,
                    }, default=float))
                row = {
                    "scenario": scenario_id, "scenario_name": spec["name"],
                    "delivery_date": date, "seed": seed,
                    "customer_count": metrics["customer_count"],
                    "vehicle_count": metrics["vehicle_count"],
                    "total_distance_km": metrics["distance_km"],
                    "total_travel_time_min": metrics["travel_time_min"],
                    "total_lateness_min": metrics["lateness_min"],
                    "on_time_count": metrics["on_time_count"],
                    "otd_pct": metrics["otd_pct"],
                    "risk_penalty_min": metrics["risk_penalty_min"],
                    "overtime_min": metrics["overtime_min"],
                    "objective": metrics["objective"],
                    "runtime_sec": runtime,
                    **audit,
                }
                all_rows.append(row)
                if len(hist):
                    hist = hist.copy(); hist["scenario"] = scenario_id
                    hist["delivery_date"] = date; hist["seed"] = seed
                    history_frames.append(hist)
                if best_pack is None or metrics["objective"] < best_pack[1]["objective"]:
                    best_pack = (routes, metrics, seed, runtime, audit)
            routes, metrics, seed, runtime, audit = best_pack
            best_routes[scenario_id][date] = routes
            frame = _scenario_route_frame(routes, inst, cfg, scenario_id, seed)
            route_frames.append(frame)

    all_runs = pd.DataFrame(all_rows)
    best_summary = (all_runs.sort_values(["scenario", "delivery_date", "objective"])
                    .groupby(["scenario", "delivery_date"], as_index=False).first())
    routes_df = pd.concat(route_frames, ignore_index=True) if route_frames else pd.DataFrame()
    history_df = (pd.concat(history_frames, ignore_index=True) if history_frames else
                  pd.DataFrame(columns=["iteration", "current_objective", "best_objective",
                                        "temperature", "destroy_operator", "repair_operator",
                                        "accepted", "scenario", "delivery_date", "seed"]))
    return routes_df, best_summary, all_runs, history_df, best_routes


def aggregate_scenario_results(best_summary: pd.DataFrame) -> pd.DataFrame:
    agg = (best_summary.groupby(["scenario", "scenario_name"], as_index=False)
           .agg(
               instance_count=("delivery_date", "nunique"),
               total_customers=("customer_count", "sum"),
               total_vehicles=("vehicle_count", "sum"),
               total_distance_km=("total_distance_km", "sum"),
               total_travel_time_min=("total_travel_time_min", "sum"),
               total_lateness_min=("total_lateness_min", "sum"),
               total_on_time=("on_time_count", "sum"),
               mean_daily_otd_pct=("otd_pct", "mean"),
               total_runtime_sec=("runtime_sec", "sum"),
               weight_violation_routes=("weight_violation_routes", "sum"),
               volume_violation_routes=("volume_violation_routes", "sum"),
               mean_weight_util_pct=("mean_weight_util_pct", "mean"),
               mean_volume_util_pct=("mean_volume_util_pct", "mean"),
           ))
    agg["aggregate_otd_pct"] = 100.0 * agg["total_on_time"] / agg["total_customers"].clip(lower=1)
    base = agg.loc[agg["scenario"] == "S0"].iloc[0] if (agg["scenario"] == "S0").any() else None
    if base is not None:
        for metric in ["total_distance_km", "total_travel_time_min", "total_lateness_min"]:
            base_value = float(base[metric])
            agg[f"change_vs_S0_{metric}_absolute"] = agg[metric] - base_value
            if abs(base_value) > EPS:
                agg[f"change_vs_S0_{metric}_pct"] = 100.0 * (agg[metric] - base_value) / base_value
            else:
                # Percentage change is mathematically undefined when the S0 baseline is zero.
                agg[f"change_vs_S0_{metric}_pct"] = np.nan
        agg["change_vs_S0_otd_pp"] = agg["aggregate_otd_pct"] - float(base["aggregate_otd_pct"])
    return agg


def export_end_to_end_results(
    output_dir: str | Path,
    repo: DataRepository,
    raw_audit: pd.DataFrame,
    coordinate_audit: pd.DataFrame,
    scenario_routes: pd.DataFrame,
    scenario_best: pd.DataFrame,
    scenario_all_runs: pd.DataFrame,
    scenario_history: pd.DataFrame,
    s5_results: pd.DataFrame,
    s5_sequences: pd.DataFrame,
    config_payload: Dict,
) -> Path:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    aggregate = aggregate_scenario_results(scenario_best)
    s4_routes = scenario_routes[scenario_routes["scenario"] == "S4"].copy()
    s4_best = scenario_best[scenario_best["scenario"] == "S4"].copy()
    s4_all = scenario_all_runs[scenario_all_runs["scenario"] == "S4"].copy()
    s4_hist = scenario_history[scenario_history["scenario"] == "S4"].copy()

    # Produce the existing S4/S5 package first.
    export_results(
        output_dir, repo, s4_routes, s4_best, s4_all, s4_hist,
        s5_results, s5_sequences, config_payload,
    )

    extra_files = {
        "raw_preprocessing_audit.csv": raw_audit,
        "coordinate_consistency_audit.csv": coordinate_audit,
        "scenario_s0_s4_routes.csv": scenario_routes,
        "scenario_s0_s4_best_by_instance.csv": scenario_best,
        "scenario_s0_s4_all_seed_runs.csv": scenario_all_runs,
        "scenario_s0_s4_iteration_history.csv": scenario_history,
        "scenario_s0_s4_aggregate_summary.csv": aggregate,
    }
    for fn, df in extra_files.items():
        df.to_csv(output_dir / fn, index=False)

    # Paper-ready comparison table.
    compare = aggregate[[
        "scenario", "scenario_name", "total_distance_km", "total_travel_time_min",
        "total_lateness_min", "aggregate_otd_pct", "total_vehicles",
        "mean_weight_util_pct", "mean_volume_util_pct", "volume_violation_routes",
    ]].copy()
    compare.to_csv(output_dir / "paper_table_s0_s4_comparison.csv", index=False)

    # Replace/add a comprehensive workbook.
    xlsx = output_dir / "End_to_End_S0_S5_Results.xlsx"
    s5_summary = summarize_s5(s5_results)
    omnibus, pairwise = statistical_tests(s5_results)
    with pd.ExcelWriter(xlsx, engine="openpyxl") as writer:
        for sheet, df in [
            ("Raw_Audit", raw_audit), ("Coordinate_Audit", coordinate_audit),
            ("Input_Matrix_Audit", repo.audit()), ("S0_S4_Aggregate", aggregate),
            ("S0_S4_Best", scenario_best), ("S0_S4_All_Runs", scenario_all_runs),
            ("S0_S4_Routes", scenario_routes), ("S5_Detail", s5_results),
            ("S5_Summary", s5_summary), ("Friedman", omnibus), ("Pairwise", pairwise),
        ]:
            df.to_excel(writer, sheet_name=sheet[:31], index=False)

    try:
        import matplotlib.pyplot as plt
        figdir = output_dir / "figures"
        figdir.mkdir(exist_ok=True)
        for metric, ylabel, fn in [
            ("total_distance_km", "Total distance (km)", "s0_s4_total_distance.png"),
            ("total_lateness_min", "Total lateness (min)", "s0_s4_total_lateness.png"),
            ("aggregate_otd_pct", "Aggregate OTD (%)", "s0_s4_otd.png"),
        ]:
            fig, ax = plt.subplots(figsize=(8, 5))
            ax.bar(aggregate["scenario"], aggregate[metric])
            ax.set_xlabel("Scenario"); ax.set_ylabel(ylabel); ax.grid(axis="y", alpha=.25)
            fig.tight_layout(); fig.savefig(figdir / fn, dpi=300); plt.close(fig)
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.scatter(aggregate["total_distance_km"], aggregate["total_lateness_min"])
        for _, r in aggregate.iterrows():
            ax.annotate(r["scenario"], (r["total_distance_km"], r["total_lateness_min"]))
        ax.set_xlabel("Total distance (km)"); ax.set_ylabel("Total lateness (min)")
        ax.grid(alpha=.25); fig.tight_layout(); fig.savefig(figdir / "s0_s4_reliability_efficiency_tradeoff.png", dpi=300); plt.close(fig)
    except Exception as e:
        (output_dir / "s0_s4_figure_generation_error.txt").write_text(str(e), encoding="utf-8")

    # Rebuild ZIP after all extra files are present.
    zip_path = output_dir.parent / (output_dir.name + ".zip")
    if zip_path.exists():
        zip_path.unlink()
    shutil.make_archive(str(zip_path.with_suffix("")), "zip", root_dir=output_dir)
    return zip_path


In [ ]:
# ============================================================
# 4. PREPROCESS RAW ORDERS AND VALIDATE SAVED MATRICES
# ============================================================
repo = None
raw_audit = pd.DataFrame()
coordinate_audit = pd.DataFrame()
matrix_audit = pd.DataFrame()

if stage in {"build_s4", "s5_batch"}:
    normalized_input, raw_audit, coordinate_audit = prepare_normalized_input_from_raw(
        RAW_XLSX_PATH, MATRIX_ZIP_PATH, WORK / "preprocessing"
    )
    display(raw_audit)
    print("Coordinate inconsistencies:",
          int((~coordinate_audit["coordinate_consistent"]).sum()))

    repo = DataRepository(
        normalized_input,
        WORK / "repository",
        capacity_weight_kg=CAPACITY_WEIGHT_KG,
        capacity_volume_cm3=CAPACITY_VOLUME_CM3,
        oversize_policy=OVERSIZE_POLICY,
    )
    matrix_audit = repo.audit()
    display(matrix_audit.head())
    print("Dates:", len(repo.dates), repo.dates[0], "to", repo.dates[-1])
    print("Customer-day stops:", len(repo.stops))


In [ ]:
# ============================================================
# 5. SHARED HELPERS
# ============================================================
def load_locked_s4_routes(csv_path, repo):
    csv_path = Path(csv_path)
    if not csv_path.exists():
        raise FileNotFoundError(
            f"Locked S4 route file not found: {csv_path}\n"
            "Run STAGE='build_s4' first."
        )
    frame = pd.read_csv(csv_path)
    required = {"delivery_date", "vehicle_id", "sequence", "node_id"}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"Locked S4 route file is missing columns: {sorted(missing)}")

    result = {}
    for date, date_frame in frame.groupby("delivery_date", sort=False):
        date = str(date)
        if date not in repo.dates:
            continue
        inst = repo.get_instance(date)
        node_to_idx = {str(node): i for i, node in enumerate(inst.node_ids)}
        routes = []
        for _, g in date_frame.sort_values(["vehicle_id", "sequence"]).groupby("vehicle_id", sort=False):
            ids = [
                str(x) for x in g["node_id"].tolist()
                if str(x) != DEPOT_ID and str(x) in node_to_idx
            ]
            route = [node_to_idx[x] for x in ids]
            if route:
                routes.append(route)
        result[date] = routes
    return result

def zip_folder(folder, zip_path):
    folder = Path(folder)
    zip_path = Path(zip_path)
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in sorted(folder.rglob("*")):
            if f.is_file() and f.resolve() != zip_path.resolve():
                zf.write(f, f.relative_to(folder))
    return zip_path

def download_if_colab(path):
    path = Path(path)
    print("Output:", path)
    if IN_COLAB:
        files.download(str(path))

def selected_batch_dates(all_dates, batch_id, batch_size):
    if batch_id < 1:
        raise ValueError("BATCH_ID starts at 1.")
    start = (batch_id - 1) * batch_size
    end = min(start + batch_size, len(all_dates))
    if start >= len(all_dates):
        raise ValueError(f"BATCH_ID={batch_id} exceeds the available dates.")
    return list(all_dates[start:end])


In [ ]:
# ============================================================
# 6A. STAGE 1 — BUILD AND LOCK S4 ONCE
# ============================================================
if stage == "build_s4":
    dates = repo.dates[:2] if QUICK_TEST else repo.dates
    iterations = 20 if QUICK_TEST else S4_ITERATIONS
    local_trials = 5 if QUICK_TEST else S4_LOCAL_SEARCH_TRIALS
    seeds = [S4_SEEDS[0]] if QUICK_TEST else S4_SEEDS

    base_cfg = S4Config(
        capacity_weight_kg=CAPACITY_WEIGHT_KG,
        capacity_volume_cm3=CAPACITY_VOLUME_CM3,
        depot_start_min=DEPOT_START_MIN,
        route_end_target_min=ROUTE_END_TARGET_MIN,
        max_routes=MAX_ROUTES_PER_DAY,
        w_vehicle=VEHICLE_OPENING_PENALTY,
        iterations=iterations,
        local_search_trials=local_trials,
        early_stop_patience=S4_EARLY_STOP,
        early_stop_min_iterations=min(20, iterations),
    )

    started = time.perf_counter()
    scenario_routes, scenario_best, scenario_all_runs, scenario_history, best_routes = (
        run_s0_s4_scenarios(
            repo=repo,
            dates=dates,
            base_cfg=base_cfg,
            seeds={"S4": seeds},
            checkpoint_dir=CHECKPOINT_S4,
            scenarios=("S4",),
            force_recompute=FORCE_RECOMPUTE_S4,
        )
    )
    runtime_sec = time.perf_counter() - started

    s4_routes = scenario_routes[scenario_routes["scenario"] == "S4"].copy()
    s4_best = scenario_best[scenario_best["scenario"] == "S4"].copy()
    s4_runs = scenario_all_runs[scenario_all_runs["scenario"] == "S4"].copy()
    s4_history = scenario_history[scenario_history["scenario"] == "S4"].copy()

    s4_routes.to_csv(LOCK_DIR / "s4_routes_best.csv", index=False)
    s4_best.to_csv(LOCK_DIR / "s4_instance_summary.csv", index=False)
    s4_runs.to_csv(LOCK_DIR / "s4_all_seed_runs.csv", index=False)
    s4_history.to_csv(LOCK_DIR / "s4_search_history.csv", index=False)

    payload = {
        "stage": "build_s4",
        "quick_test": QUICK_TEST,
        "dates_completed": len(dates),
        "expected_dates": len(repo.dates),
        "seeds": seeds,
        "iterations": iterations,
        "local_search_trials": local_trials,
        "runtime_sec": runtime_sec,
        "stability_weight_reserved_for_s5": STABILITY_WEIGHT_FINAL,
    }
    (LOCK_DIR / "s4_lock_config.json").write_text(
        json.dumps(payload, indent=2, default=float), encoding="utf-8"
    )

    display(s4_best.head())
    print(f"S4 lock completed for {len(dates)} dates in {runtime_sec/60:.2f} minutes.")
    if not QUICK_TEST and len(dates) != 63:
        print("WARNING: the expected paper dataset contains 63 dates.")

    zip_path = zip_folder(LOCK_DIR, LOCAL_OUTPUT / "QRG_v5_S4_Locked_Routes.zip")
    download_if_colab(zip_path)


In [ ]:
# ============================================================
# 6B. STAGE 2 — RUN ONE S5 BATCH
# ============================================================
if stage == "s5_batch":
    locked_file = LOCK_DIR / "s4_routes_best.csv"
    all_locked_routes = load_locked_s4_routes(locked_file, repo)

    missing_locked = [d for d in repo.dates if d not in all_locked_routes]
    if missing_locked:
        raise ValueError(
            f"Locked S4 routes are incomplete. Missing {len(missing_locked)} dates, "
            f"including: {missing_locked[:5]}"
        )

    batch_dates = selected_batch_dates(repo.dates, BATCH_ID, BATCH_SIZE)
    print(f"Batch {BATCH_ID}: {batch_dates[0]} to {batch_dates[-1]} "
          f"({len(batch_dates)} dates)")

    if QUICK_TEST:
        batch_dates = batch_dates[:1]
        s5_seeds = [S5_SEEDS[0]]
        s5_triggers = (0.50,)
        s5_shocks = ("p95", "delay20")
        s5_iterations = 10
    else:
        s5_seeds = S5_SEEDS
        s5_triggers = S5_TRIGGERS
        s5_shocks = S5_SHOCKS
        s5_iterations = S5_ITERATIONS

    s5_cfg = S5Config(
        depot_start_min=DEPOT_START_MIN,
        route_end_target_min=ROUTE_END_TARGET_MIN,
        trigger_fractions=s5_triggers,
        shocks=s5_shocks,
        methods=S5_METHODS,
        iterations_fr=s5_iterations,
        iterations_rg=s5_iterations,
        iterations_sa=s5_iterations,
        stability_weight=float(STABILITY_WEIGHT_FINAL),
        early_stop_patience=S5_EARLY_STOP,
        early_stop_min_iterations=min(10, s5_iterations),
    )

    routes_for_batch = {d: all_locked_routes[d] for d in batch_dates}
    started = time.perf_counter()
    s5_results, s5_sequences = run_s5(
        repo=repo,
        dates=batch_dates,
        best_routes_by_date=routes_for_batch,
        cfg=s5_cfg,
        seeds=s5_seeds,
        checkpoint_dir=CHECKPOINT_S5,
    )
    runtime_sec = time.perf_counter() - started

    batch_tag = f"{BATCH_ID:02d}"
    result_path = BATCH_DIR / f"s5_batch_{batch_tag}_results.csv"
    sequence_path = BATCH_DIR / f"s5_batch_{batch_tag}_sequences.csv"
    config_path = BATCH_DIR / f"s5_batch_{batch_tag}_config.json"

    s5_results.to_csv(result_path, index=False)
    s5_sequences.to_csv(sequence_path, index=False)
    config_path.write_text(json.dumps({
        "batch_id": BATCH_ID,
        "batch_size": BATCH_SIZE,
        "dates": batch_dates,
        "seeds": s5_seeds,
        "triggers": s5_triggers,
        "shocks": s5_shocks,
        "methods": S5_METHODS,
        "iterations": s5_iterations,
        "early_stop": S5_EARLY_STOP,
        "stability_weight": STABILITY_WEIGHT_FINAL,
        "runtime_sec": runtime_sec,
        "quick_test": QUICK_TEST,
    }, indent=2, default=float), encoding="utf-8")

    display(summarize_s5(s5_results))
    print(f"Batch {BATCH_ID} completed in {runtime_sec/60:.2f} minutes.")

    batch_package = LOCAL_OUTPUT / f"QRG_v5_S5_Batch_{batch_tag}.zip"
    with zipfile.ZipFile(batch_package, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in [result_path, sequence_path, config_path]:
            zf.write(p, p.name)
    download_if_colab(batch_package)


In [ ]:
# ============================================================
# 6C. STAGE 3 — MERGE ALL S5 BATCHES AND EXPORT PAPER TABLES
# ============================================================
if stage == "merge":
    result_files = sorted(BATCH_DIR.glob("s5_batch_*_results.csv"))
    sequence_files = sorted(BATCH_DIR.glob("s5_batch_*_sequences.csv"))
    if not result_files:
        raise FileNotFoundError(
            f"No S5 batch result files found in {BATCH_DIR}. "
            "Complete STAGE='s5_batch' first."
        )

    s5_results = pd.concat([pd.read_csv(p) for p in result_files], ignore_index=True)
    s5_sequences = (
        pd.concat([pd.read_csv(p) for p in sequence_files], ignore_index=True)
        if sequence_files else pd.DataFrame()
    )

    key_cols = ["delivery_date", "trigger_fraction", "shock", "seed", "method"]
    s5_results = s5_results.drop_duplicates(key_cols, keep="last")
    if len(s5_sequences):
        sequence_keys = key_cols + ["vehicle_id", "sequence"]
        s5_sequences = s5_sequences.drop_duplicates(sequence_keys, keep="last")

    completed_dates = sorted(s5_results["delivery_date"].astype(str).unique())
    expected_date_count = 63
    print("Completed dates:", len(completed_dates))
    print("Expected dates:", expected_date_count)

    expected_rows_per_date = (
        len(S5_TRIGGERS) * len(S5_SHOCKS) * len(S5_SEEDS) * len(S5_METHODS)
    )
    counts = s5_results.groupby("delivery_date").size()
    incomplete = counts[counts != expected_rows_per_date]

    if len(completed_dates) != expected_date_count or len(incomplete):
        print("WARNING: merged output is incomplete.")
        if len(incomplete):
            display(incomplete.rename("rows_found").reset_index())
        print("Run the missing batch(es) before using these results in the paper.")

    summary = summarize_s5(s5_results)
    omnibus, pairwise = statistical_tests(s5_results)

    # Additional paper-facing summaries.
    method_summary = (
        s5_results.groupby("method", as_index=False)
        .agg(
            n_blocks=("delivery_date", "size"),
            mean_lateness_min=("total_lateness_min", "mean"),
            median_lateness_min=("total_lateness_min", "median"),
            mean_otd_pct=("otd_pct", "mean"),
            mean_lateness_reduction_min=("lateness_reduction_min", "mean"),
            no_harm_rate_pct=("replanning_avoids_new_lateness", lambda x: 100 * x.mean()),
            mean_route_stability=("route_stability_index", "mean"),
            mean_moved_stops=("moved_stops", "mean"),
            mean_distance_premium_pct=("distance_premium_pct", "mean"),
            mean_runtime_sec=("runtime_sec", "mean"),
        )
    )
    condition_summary = (
        s5_results.groupby(["method", "shock", "trigger_fraction"], as_index=False)
        .agg(
            n_blocks=("delivery_date", "size"),
            mean_lateness_min=("total_lateness_min", "mean"),
            mean_lateness_reduction_min=("lateness_reduction_min", "mean"),
            mean_otd_improvement_pp=("otd_improvement_pp", "mean"),
            no_harm_rate_pct=("replanning_avoids_new_lateness", lambda x: 100 * x.mean()),
            mean_route_stability=("route_stability_index", "mean"),
            mean_moved_stops=("moved_stops", "mean"),
        )
    )

    FINAL_DIR.mkdir(parents=True, exist_ok=True)
    s5_results.to_csv(FINAL_DIR / "s5_results_final.csv", index=False)
    s5_sequences.to_csv(FINAL_DIR / "s5_sequences_final.csv", index=False)
    summary.to_csv(FINAL_DIR / "s5_summary_by_condition.csv", index=False)
    method_summary.to_csv(FINAL_DIR / "s5_method_summary.csv", index=False)
    condition_summary.to_csv(FINAL_DIR / "s5_condition_summary.csv", index=False)
    omnibus.to_csv(FINAL_DIR / "s5_friedman_test.csv", index=False)
    pairwise.to_csv(FINAL_DIR / "s5_pairwise_wilcoxon_holm.csv", index=False)

    lock_files = [
        LOCK_DIR / "s4_routes_best.csv",
        LOCK_DIR / "s4_instance_summary.csv",
        LOCK_DIR / "s4_lock_config.json",
    ]
    for p in lock_files:
        if p.exists():
            shutil.copy2(p, FINAL_DIR / p.name)

    manifest = {
        "completed_dates": len(completed_dates),
        "expected_dates": expected_date_count,
        "expected_rows_per_date": expected_rows_per_date,
        "total_result_rows": len(s5_results),
        "batch_result_files": [p.name for p in result_files],
        "stability_weight": STABILITY_WEIGHT_FINAL,
        "s5_seeds": S5_SEEDS,
        "triggers": S5_TRIGGERS,
        "shocks": S5_SHOCKS,
        "methods": S5_METHODS,
        "iterations": S5_ITERATIONS,
        "ready_for_paper": bool(
            len(completed_dates) == expected_date_count and len(incomplete) == 0
        ),
    }
    (FINAL_DIR / "paper_final_manifest.json").write_text(
        json.dumps(manifest, indent=2, default=float), encoding="utf-8"
    )

    display(method_summary)
    display(omnibus)
    display(pairwise)

    final_zip = zip_folder(FINAL_DIR, LOCAL_OUTPUT / "QRG_v5_Paper_Final_Output.zip")
    download_if_colab(final_zip)


## Exact upload-only execution order

1. Set `STAGE = "build_s4"`; upload the raw Excel and matrix ZIP; download `QRG_v5_S4_Locked_Routes.zip`.
2. Set `STAGE = "s5_batch"`; for each `BATCH_ID = 1` through `7`, upload the raw Excel, matrix ZIP, and the S4 locked-route ZIP; download each batch ZIP.
3. Set `STAGE = "merge"`; upload the S4 locked-route ZIP and all seven batch ZIPs; download `QRG_v5_Paper_Final_Output.zip`.

Within one active Colab runtime, after the first S5 batch you may change `BATCH_ID` and rerun only the S5 batch cell, so the three inputs do not need to be uploaded again. If the runtime disconnects, upload them again.

Because there is no Google Drive persistence, keep every downloaded ZIP. Do not use the manuscript results unless `paper_final_manifest.json` reports `"ready_for_paper": true`.
